In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.2 MB/s eta 0:00:00


In [3]:
import os

# --- OOM fix: reduce CUDA memory fragmentation. Must be set before torch is imported. ---
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Required for torch.use_deterministic_algorithms(True) further down (set_full_seed)
# to work with CUDA matmul/conv kernels -- must also be set before torch is imported.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SMRI_FEATURES_CSV = "/content/drive/MyDrive/DHGFormer/abide_smri.csv"
PHENOTYPIC_CSV    = "/content/drive/MyDrive/DHGFormer/Phenotypic_V1_0b_preprocessed1.csv"

save_path = "/content/drive/MyDrive/sMRI"
os.makedirs(save_path, exist_ok=True)

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.metrics import roc_curve, auc
import pandas as pd
import os
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.feature_selection import RFE

def feature_selection(matrix, labels, train_ind, fnum):
    """
        matrix       : feature matrix (num_subjects x num_features)
        labels       : ground truth labels (num_subjects x 1)
        train_ind    : indices of the training samples
        fnum         : size of the feature vector after feature selection

    return:
        x_data      : feature matrix of lower dimension (num_subjects x fnum)
    """

    estimator = RidgeClassifier()
    selector = RFE(estimator, n_features_to_select=fnum, step=100, verbose=1)

    featureX = matrix[train_ind, :]
    featureY = labels[train_ind]
    selector = selector.fit(featureX, featureY.ravel())

    return selector

def get_index(lst=None, item=''):
	return [i for i in range(len(lst)) if lst[i] == item]

def save_model(net,path, name_net):

  # This fucntion is used to save a specific model
  # (fixed to work on CPU-only machines: only moves back to GPU if it was there)

    os.makedirs(path, exist_ok=True)   # create `path` (and any missing parents) if it doesn't exist yet
    path_net =  path + '/' + name_net + '.pth'
    was_cuda = next(net.parameters()).is_cuda
    torch.save(net.cpu().state_dict(), path_net)
    if was_cuda:
        net.cuda()


In [5]:
########################################### Load Data ###############################################
#####################################################################################################
#####################################################################################################
combat = False    # True or False

if combat == False:
  save_combat = 'without_ComBat'
else:
  save_combat = 'with_ComBat'

save_path = os.path.join(save_path, save_combat)
os.makedirs(save_path, exist_ok=True)

k_fold = 5
scaler = True

# CPU / GPU are both supported automatically from here on (see args.cuda below,
# which is derived from torch.cuda.is_available()).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

########################################### Phenotypic table ###############################################
# Set this to True only if you specifically want the original authors' QC subsample
# (SUB_IN_SMP == 1). Default is False so you keep every subject you have features for.
APPLY_SUB_IN_SMP_FILTER = False

pheno_df = pd.read_csv(PHENOTYPIC_CSV)
pheno_df['FILE_ID'] = pheno_df['FILE_ID'].astype(str)
print(f'Phenotypic file: {len(pheno_df)} rows total')

pheno_df = pheno_df[pheno_df['FILE_ID'] != 'no_filename']   # these rows have no scan at all, so they can never match a features row anyway
print(f'  -> {len(pheno_df)} rows have a FILE_ID (i.e. an actual scan)')

if APPLY_SUB_IN_SMP_FILTER and 'SUB_IN_SMP' in pheno_df.columns:
  pheno_df = pheno_df[pheno_df['SUB_IN_SMP'] == 1]
  print(f'  -> {len(pheno_df)} rows remain after the SUB_IN_SMP QC filter')

########################################### sMRI feature table ###############################################
smri_df = pd.read_csv(SMRI_FEATURES_CSV)
smri_df['subject_id'] = smri_df['subject_id'].astype(str)
print(f'sMRI features file: {len(smri_df)} subjects')

# only keep subjects present in BOTH files
ids_pheno = set(pheno_df['FILE_ID'])
ids_smri = set(smri_df['subject_id'])
common_ids = sorted(ids_pheno & ids_smri)
only_in_pheno = ids_pheno - ids_smri
only_in_smri = ids_smri - ids_pheno
print(f'{len(common_ids)} subjects found in both files')
if only_in_pheno:
  print(f"  {len(only_in_pheno)} subject(s) are in the phenotypic file but have no matching row in the sMRI features file (no imaging features, so they cannot be used), e.g.: {sorted(only_in_pheno)[:5]}")
if only_in_smri:
  print(f"  {len(only_in_smri)} subject(s) are in the sMRI features file but have no matching row in the phenotypic file (no label/site/age, so they cannot be used), e.g.: {sorted(only_in_smri)[:5]}")

pheno_df = pheno_df.set_index('FILE_ID').loc[common_ids]
smri_df = smri_df.set_index('subject_id').loc[common_ids]

subject_IDs = list(common_ids)
number_samples = len(subject_IDs)

sites = pheno_df['SITE_ID'].astype(str).str.replace(' ', '').tolist()
unique_sites = list(np.unique(sites))

# DX_GROUP: 1 = Autism, 2 = Control (standard ABIDE convention)  ->  labels: 1 = Autism, 0 = Control
labels = (pheno_df['DX_GROUP'].values == 1).astype(np.float64)


Using device: cuda
Phenotypic file: 1112 rows total
  -> 1035 rows have a FILE_ID (i.e. an actual scan)
sMRI features file: 1009 subjects
1009 subjects found in both files
  26 subject(s) are in the phenotypic file but have no matching row in the sMRI features file (no imaging features, so they cannot be used), e.g.: ['OHSU_0050142', 'OHSU_0050143', 'OHSU_0050144', 'OHSU_0050145', 'OHSU_0050146']


In [6]:
# ############################################### Load sMRI (flat feature matrix) ##########################################
# ############################################### Load sMRI (flat feature matrix) ##########################################
# ############################################### Load sMRI (flat feature matrix) ##########################################
# # The original notebook parsed FreeSurfer .stats files subject-by-subject and built
# # the Desikan-Killiany / aseg / wmparc blocks by hand. All of that is now already
# # done for us inside SMRI_FEATURES_CSV (one row per subject, one column per
# # feature), so we just load it directly.
# #
# # This flat matrix feeds the Ridge/RFE feature_selection() step in the next cell.
# # The GCN's per-ROI-node features (built in the "Load sMRI (per-view ROI graphs)"
# # section further down) are built straight from smri_df, but they now *do* respect
# # the Ridge/RFE result too -- see USE_RIDGE_FEATURE_SELECTION in the next cell.
# from sklearn.impute import SimpleImputer, KNNImputer

# feature_cols = list(smri_df.columns)
# sMRI_features = smri_df[feature_cols].apply(pd.to_numeric, errors='coerce').values.astype(np.float64)

# ########################################### Fill missing values (NaN) BEFORE Ridge/RFE ###########################################
# # IMPORTANT: imputation is done before the Ridge/RFE step so RidgeClassifier never
# # receives NaN values. We use the same column-mean strategy already used later in
# # the per-view graph construction, so the downstream feature values/rules remain
# # unchanged apart from making the Ridge input finite.
# nan_count = int(np.isnan(sMRI_features).sum())
# print(f'Filling {nan_count} missing values with column means before Ridge/RFE')

# imputer = SimpleImputer(strategy='mean')
# sMRI_features = imputer.fit_transform(sMRI_features)

# # Keep the existing scaling order: imputation -> scaling -> Ridge/RFE.
# # This is important because Ridge is sensitive to feature scale.
# if scaler == True:
#   sMRI_features = StandardScaler().fit_transform(sMRI_features)

# print('sMRI feature matrix shape:', sMRI_features.shape)


########################################### Load sMRI (flat feature matrix) ###########################################

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

feature_cols = list(smri_df.columns)

# Raw numeric sMRI feature matrix
sMRI_features = (
    smri_df[feature_cols]
    .apply(pd.to_numeric, errors='coerce')
    .values
    .astype(np.float64)
)

# if scaler == True:
#     scaler_obj = StandardScaler()
#     sMRI_features = scaler_obj.fit_transform(sMRI_features)

print('Raw sMRI feature matrix shape:', sMRI_features.shape)
print('NaNs before imputation:', np.isnan(sMRI_features).sum())


# ------------------------------------------------------------------
# Imputation MUST happen before Ridge/RFE
# ------------------------------------------------------------------

IMPUTE_STRATEGY = 'knn'
KNN_NEIGHBORS = 10


def make_imputer(strategy):
    if strategy == 'mean':
        return SimpleImputer(strategy='mean')
    elif strategy == 'median':
        return SimpleImputer(strategy='median')
    elif strategy == 'knn':
        return KNNImputer(
            n_neighbors=KNN_NEIGHBORS,
            weights='distance'
        )
    elif strategy == 'iterative':
        return IterativeImputer(
            max_iter=10,
            random_state=0
        )
    else:
        raise ValueError(f'Unknown strategy: {strategy}')


# Create imputer
imputer = make_imputer(IMPUTE_STRATEGY)

# Fill NaNs
sMRI_features = imputer.fit_transform(sMRI_features)

print('NaNs after imputation:', np.isnan(sMRI_features).sum())


# ------------------------------------------------------------------
# Scaling AFTER imputation
# ------------------------------------------------------------------

if scaler == True:
    scaler_obj = StandardScaler()
    sMRI_features = scaler_obj.fit_transform(sMRI_features)

print('Final sMRI feature matrix shape:', sMRI_features.shape)

Raw sMRI feature matrix shape: (1009, 2386)
NaNs before imputation: 594
NaNs after imputation: 0
Final sMRI feature matrix shape: (1009, 2386)


In [7]:
# ########################################### Feature selection (configurable) ###########################################
# # NUM_FEATURES_TO_SELECT: set this to an integer to run the Ridge-based RFE
# # feature_selection() (defined earlier) and keep only that many sMRI features.
# # Leave it as None to skip Ridge/RFE entirely (fastest option; nothing filtered).
# #
# # USE_RIDGE_FEATURE_SELECTION: this is what actually makes the Ridge/RFE result
# # matter. When True, the per-view ROI graphs built in the next cell only use the
# # sMRI columns Ridge/RFE selected here -- every other column is treated as
# # "missing" for that node (same missing -> filled-with-0 rule already used for
# # columns that don't exist at all in a given view). Graph topology, node count,
# # and model architecture are all untouched either way.
# #
# # NaNs are filled in the previous cell before Ridge/RFE runs. The selected
# # feature indices are then applied exactly as before; graph topology, node count,
# # and model architecture are unchanged.
# NUM_FEATURES_TO_SELECT = 500       # <-- set to an int (e.g. 500) to run Ridge/RFE and select that many features
# USE_RIDGE_FEATURE_SELECTION = False  # <-- set True to actually apply that selection to the GCN's per-view features

# if NUM_FEATURES_TO_SELECT is None:
#   new_number_features = sMRI_features.shape[1]
#   selected_feature_idx = None
# else:
#   new_number_features = NUM_FEATURES_TO_SELECT
#   all_indices = np.arange(sMRI_features.shape[0])
#   rfe_selector = feature_selection(sMRI_features, labels, all_indices, new_number_features)
#   selected_feature_idx = np.where(rfe_selector.support_)[0]
#   print(f'Ridge RFE selected {len(selected_feature_idx)} feature(s) out of {sMRI_features.shape[1]}')

# if USE_RIDGE_FEATURE_SELECTION and selected_feature_idx is None:
#   raise ValueError(
#       'USE_RIDGE_FEATURE_SELECTION=True requires NUM_FEATURES_TO_SELECT to be an '
#       'int, so Ridge/RFE actually runs and produces a selection to apply.'
#   )

# # Set of flat-matrix column indices Ridge/RFE kept (None if Ridge/RFE didn't run).
# selected_feature_idx_set = set(selected_feature_idx.tolist()) if selected_feature_idx is not None else None
# # Maps a raw sMRI column name -> its position in feature_cols / sMRI_features, so
# # the per-view graph cell below can look up whether Ridge/RFE kept that column.
# feature_col_to_idx = {name: idx for idx, name in enumerate(feature_cols)}

# print('Final feature count (new_number_features):', new_number_features)
# print('Ridge/RFE selection actually applied to GCN per-view features:', USE_RIDGE_FEATURE_SELECTION)

########################################### Feature selection (configurable) ###########################################

NUM_FEATURES_TO_SELECT = 500
USE_RIDGE_FEATURE_SELECTION = True

if NUM_FEATURES_TO_SELECT is None:

    new_number_features = sMRI_features.shape[1]
    selected_feature_idx = None

else:

    new_number_features = NUM_FEATURES_TO_SELECT

    all_indices = np.arange(sMRI_features.shape[0])

    rfe_selector = feature_selection(
        sMRI_features,
        labels,
        all_indices,
        new_number_features
    )

    selected_feature_idx = np.where(rfe_selector.support_)[0]

    print(
        f'Ridge RFE selected {len(selected_feature_idx)} '
        f'feature(s) out of {sMRI_features.shape[1]}'
    )


if USE_RIDGE_FEATURE_SELECTION and selected_feature_idx is None:
    raise ValueError(
        'USE_RIDGE_FEATURE_SELECTION=True requires '
        'NUM_FEATURES_TO_SELECT to be an int, so Ridge/RFE '
        'actually runs and produces a selection to apply.'
    )


selected_feature_idx_set = (
    set(selected_feature_idx.tolist())
    if selected_feature_idx is not None
    else None
)

feature_col_to_idx = {
    name: idx
    for idx, name in enumerate(feature_cols)
}


print(
    'Final feature count (new_number_features):',
    new_number_features
)

print(
    'Ridge/RFE selection actually applied to GCN per-view features:',
    USE_RIDGE_FEATURE_SELECTION
)




Fitting estimator with 2386 features.
Fitting estimator with 2286 features.
Fitting estimator with 2186 features.
Fitting estimator with 2086 features.
Fitting estimator with 1986 features.
Fitting estimator with 1886 features.
Fitting estimator with 1786 features.
Fitting estimator with 1686 features.
Fitting estimator with 1586 features.
Fitting estimator with 1486 features.
Fitting estimator with 1386 features.
Fitting estimator with 1286 features.
Fitting estimator with 1186 features.
Fitting estimator with 1086 features.
Fitting estimator with 986 features.
Fitting estimator with 886 features.
Fitting estimator with 786 features.
Fitting estimator with 686 features.
Fitting estimator with 586 features.
Ridge RFE selected 500 feature(s) out of 2386
Final feature count (new_number_features): 500
Ridge/RFE selection actually applied to GCN per-view features: True


In [8]:
# The sMRI feature table is the concatenation of 3 separate FreeSurfer outputs:
# aseg (subcortical ROI volumes), aparc left+right hemisphere (cortical ROI stats,
# columns prefixed lh_/rh_), and wmparc (white matter ROI stats). Column names look
# like "<prefix>_<ROI name>_<sub-feature>", e.g. "aseg_3rd-Ventricle_Volume_mm3" or
# "lh_G_and_S_cingul-Ant_ThickAvg". Each ROI becomes one graph node; the graph for
# each view has one node per brain region in that view, not one node per subject.
#
# If USE_RIDGE_FEATURE_SELECTION is True (set in the previous cell), any sub-feature
# column Ridge/RFE did NOT select is skipped below and left as NaN, which the
# existing "fully-empty column -> 0" handling further down already zeroes out --
# so a non-selected sub-feature simply contributes nothing to that node, without
# changing the number of nodes or the model architecture.

ASEG_STYLE_SUFFIXES = ['NVoxels', 'Volume_mm3', 'normMax', 'normMean', 'normMin', 'normRange', 'normStdDev']
APARC_STYLE_SUFFIXES = ['NumVert', 'SurfArea', 'GrayVol', 'ThickAvg', 'ThickStd', 'MeanCurv', 'GausCurv', 'FoldInd', 'CurvInd']

VIEW_CONFIGS = {
    'aseg':   {'prefixes': ['aseg'],     'suffixes': ASEG_STYLE_SUFFIXES},
    'aparc':  {'prefixes': ['lh', 'rh'], 'suffixes': APARC_STYLE_SUFFIXES},
    'wmparc': {'prefixes': ['wmparc'],   'suffixes': ASEG_STYLE_SUFFIXES},
}
view_names = list(VIEW_CONFIGS.keys())


def parse_roi_columns(prefix, columns, suffixes):
    """Groups '<prefix>_<roi>_<suffix>' columns by roi name. Columns that match no
    known suffix (e.g. wmparc's global 'Measure_...' columns) are skipped, since
    they don't belong to a single brain region."""
    sorted_suffixes = sorted(suffixes, key=len, reverse=True)
    roi_map = {}
    for col in columns:
        if not col.startswith(prefix + '_'):
            continue
        remainder = col[len(prefix) + 1:]
        for suf in sorted_suffixes:
            if remainder.endswith('_' + suf):
                roi = remainder[: -(len(suf) + 1)]
                roi_map.setdefault(roi, {})[suf] = col
                break
    return roi_map


view_node_names = {}     # view -> list of brain-region names (the graph nodes)
view_node_features = {}  # view -> np.array shape (n_subjects, n_nodes, n_subfeat)

for view, cfg in VIEW_CONFIGS.items():
    roi_entries = []
    multi_prefix = len(cfg['prefixes']) > 1
    for prefix in cfg['prefixes']:
        roi_map = parse_roi_columns(prefix, smri_df.columns, cfg['suffixes'])
        for roi_name, suf_to_col in roi_map.items():
            node_name = f'{prefix}_{roi_name}' if multi_prefix else roi_name
            col_list = [suf_to_col.get(suf) for suf in cfg['suffixes']]
            roi_entries.append((node_name, col_list))

    n_nodes = len(roi_entries)
    n_subfeat = len(cfg['suffixes'])
    mat = np.full((number_samples, n_nodes, n_subfeat), np.nan, dtype=np.float64)

    n_dropped_by_ridge = 0
    for node_idx, (node_name, col_list) in enumerate(roi_entries):
        for suf_idx, col_name in enumerate(col_list):
            if col_name is None:
                continue
            if USE_RIDGE_FEATURE_SELECTION and feature_col_to_idx.get(col_name) not in selected_feature_idx_set:
                n_dropped_by_ridge += 1
                continue   # not selected by Ridge/RFE -> left as NaN, zeroed out by the all-NaN-column rule below
            mat[:, node_idx, suf_idx] = sMRI_features[:, feature_col_to_idx[col_name]]

    if USE_RIDGE_FEATURE_SELECTION and n_dropped_by_ridge:
        print(f'  view "{view}": {n_dropped_by_ridge} sub-feature column(s) zeroed out (not selected by Ridge/RFE)')


    flat = mat.reshape(number_samples, -1)
    all_nan_cols = np.all(np.isnan(flat), axis=0)

    if all_nan_cols.any():
        flat[:, all_nan_cols] = 0.0                 # same rule as before: fully empty -> zero

    # NaN imputation already happened once, in the previous cell (on sMRI_features).
    # Every value pulled into `mat` above already comes from that imputed array, so
    # there is nothing left to impute here -- only the all-NaN columns handled above.

    mat = flat.reshape(number_samples, n_nodes, n_subfeat)

    if scaler:
        flat = StandardScaler().fit_transform(mat.reshape(number_samples, -1))
        mat = flat.reshape(number_samples, n_nodes, n_subfeat)

    view_node_names[view] = [name for name, _ in roi_entries]
    view_node_features[view] = mat.astype(np.float32)
    print(f'view "{view}": {n_nodes} brain-region nodes x {n_subfeat} sub-features per node')

# Node features are identical for every fold (no per-fold feature selection needed
# any more, since each node already only carries its own small set of sub-features).
# So the "stack every subject's graph into one big batched graph" layout only needs
# to be built once, here, and reused by every fold below.
BATCH_VEC = {}     # view -> which subject each row of X_BATCHED[view] belongs to
X_BATCHED = {}      # view -> (n_subjects * n_nodes, n_subfeat)
for view in view_names:
    n_nodes = view_node_features[view].shape[1]
    BATCH_VEC[view] = np.repeat(np.arange(number_samples), n_nodes).astype(np.int64)
    X_BATCHED[view] = view_node_features[view].reshape(number_samples * n_nodes, -1)


  view "aseg": 268 sub-feature column(s) zeroed out (not selected by Ridge/RFE)
view "aseg": 45 brain-region nodes x 7 sub-features per node
  view "aparc": 999 sub-feature column(s) zeroed out (not selected by Ridge/RFE)
view "aparc": 148 brain-region nodes x 9 sub-features per node
  view "wmparc": 374 sub-feature column(s) zeroed out (not selected by Ridge/RFE)
view "wmparc": 70 brain-region nodes x 7 sub-features per node


In [9]:
dist_train = {}
dist_validation = {}
dist_test = {}
for i in range(k_fold):
	dist_train[str(i + 1)] = []
	dist_validation[str(i + 1)] = []
	dist_test[str(i + 1)] = []

for each_site in unique_sites:
	index_site = get_index(sites, each_site)
	label = np.zeros((len(index_site)))
	for i in range(len(index_site)):
		index = index_site[i]
		label[i] = int(labels[int(index)])
	########################################### StratifiedKFold ####################################################
	sfolder = StratifiedKFold(n_splits=k_fold,random_state=0,shuffle=True)
	group = 0
	for train, validation in sfolder.split(index_site,label):
		for i in train:
			dist_train[str(group + 1)].append(index_site[i])
		for j in validation:
			dist_validation[str(group + 1)].append(index_site[j])
		group = group+1

	group = 0
	for train, validation in sfolder.split(index_site,label):
		if group == 0:
			for j in validation:
				dist_test[str(group + k_fold)].append(index_site[j])
				dist_train[str(group + k_fold)].remove(index_site[j])
		else:
			for j in validation:
				dist_test[str(group)].append(index_site[j])
				dist_train[str(group)].remove(index_site[j])
		group = group+1


In [10]:
import torch.nn as nn
import torch_geometric as tg
from torch_geometric.utils import remove_self_loops, add_self_loops, scatter as pyg_scatter
try:
    from torch_geometric.utils import maybe_num_nodes
except ImportError:  # older/newer torch_geometric versions keep this helper elsewhere
    from torch_geometric.utils.num_nodes import maybe_num_nodes

############################################### Multiview GCN model ###############################################################
############################################### Multiview GCN model ###############################################################
############################################### Multiview GCN model ###############################################################
# Each view is its own brain-region graph (nodes = ROIs of that view, shared by
# every subject; only the node features differ per subject). One shallow ChebConv
# per view produces a per-node embedding, global_mean_pool turns that into one
# embedding per subject, then the 3 view embeddings are concatenated and passed
# through a single linear classifier. Kept shallow (1 conv per view, small hidden
# size, dropout everywhere) on purpose to limit overfitting.
#
# graph_mode='static'           : same behavior as before, unchanged -- edge_weight
#                                  is the fixed partial-correlation weight computed
#                                  outside the model (in build_covariance_graph) and
#                                  passed in as-is on every call.
# graph_mode='learnable'        : each view's edge WEIGHTS become an nn.Parameter,
#                                  but they are *initialized* to the current
#                                  partial-correlation values, so epoch 1 starts out
#                                  equivalent to the static case and training only
#                                  fine-tunes that prior.
# graph_mode='learnable_scratch': same as 'learnable' -- edge topology (which nodes
#                                  are connected, from the kNN partial-correlation
#                                  graph) still comes from build_covariance_graph --
#                                  but the edge WEIGHTS are no longer initialized
#                                  from that prior. They start from a small random
#                                  value and are learned entirely from data, so the
#                                  model isn't handed any pre-computed correlation
#                                  strength to begin with, only which regions are
#                                  allowed to talk to each other.
#
# In every learnable mode, because the weights are nn.Parameter and a submodule of
# the model itself, the optimizer (built from model.parameters()) updates them
# automatically via backprop -- no change needed to the train loop/optimizer.

############################################### Message-passing layer type (conv_type) ############################################
############################################### Message-passing layer type (conv_type) ############################################
############################################### Message-passing layer type (conv_type) ############################################
# Which GNN layer runs inside each view branch. Configured from the notebook via
# MultiViewGCN(..., conv_type=...). See the CONV_TYPE cell below (next to
# GRAPH_MODE) to change it.
#
#   'cheb'  : Chebyshev spectral conv (SafeChebConv, defined above) -- THIS IS
#             THE ORIGINAL / DEFAULT PATH, byte-for-byte unchanged. Leaving
#             conv_type='cheb' reproduces exactly the results you were already
#             getting; nothing else about the model changes.
#   'gcn'   : standard GCN (Kipf & Welling). Uses edge_weight directly, with
#             the normal (non-'safe') symmetric degree normalization, so with
#             graph_mode='learnable'/'learnable_scratch' it can in principle
#             hit the same negative-degree NaN edge case that SafeChebConv was
#             built to avoid -- the fix above is specific to ChebConv.
#   'graph' : GraphConv (Morris et al.) -- weighted sum aggregation, NO
#             Laplacian/degree normalization at all, so it never divides by a
#             (possibly signed / near-zero) degree. The safest simple choice
#             if you want to pair signed, learnable edge weights with
#             something other than ChebConv.
#   'gat'   : Graph Attention. There is no scalar "edge weight" in GAT the way
#             there is for the others -- edge_weight is fed in as a 1-dim
#             edge feature (edge_dim=1) that the attention mechanism
#             conditions on, so think of it as attention *guided by*
#             correlation rather than weighted message passing.
#   'gin'   : GIN (Xu et al.), via GINEConv so it can still consume the edge
#             weight as a 1-dim edge feature (edge_dim=1), summed into each
#             neighbor's message before the MLP. Plain GINConv has no edge
#             input at all and would silently ignore edge weights entirely.
#   'sage'  : GraphSAGE. PyG's SAGEConv takes no edge_weight/edge_attr input
#             of any kind -- every neighbor is aggregated unweighted (mean),
#             so graph_mode only ever affects which edges exist (topology),
#             never their weight.
CONV_TYPES = ('cheb', 'gcn', 'graph', 'gat', 'gin', 'sage', 'tag', 'sgc', 'arma', 'bern')
GAT_HEADS = 1   # only used when conv_type == 'gat'; heads are averaged (concat=False) so output stays hid_c


def build_view_conv(conv_type, in_c, hid_c, K):
    """Builds the single message-passing layer for one view branch.

    conv_type='cheb' returns EXACTLY the same SafeChebConv(...) call that was
    hardcoded here before -- that path is untouched, so the default keeps
    reproducing your original results. Every other conv_type is a new,
    opt-in alternative.
    """
    if conv_type == 'cheb':
        return SafeChebConv(in_c, hid_c, K, normalization='sym', bias=True)
    elif conv_type == 'gcn':
        return tg.nn.GCNConv(in_c, hid_c, add_self_loops=True, normalize=True, bias=True)
    elif conv_type == 'graph':
        return tg.nn.GraphConv(in_c, hid_c, aggr='add', bias=True)
    elif conv_type == 'gat':
        return tg.nn.GATConv(in_c, hid_c, heads=GAT_HEADS, concat=False,
                              edge_dim=1, dropout=0.0, bias=True)
    elif conv_type == 'gin':
        mlp = nn.Sequential(nn.Linear(in_c, hid_c), nn.ReLU(), nn.Linear(hid_c, hid_c))
        return tg.nn.GINEConv(mlp, edge_dim=1)
    elif conv_type == 'sage':
        return tg.nn.SAGEConv(in_c, hid_c, aggr='mean', bias=True)
    elif conv_type == 'sgc':
        return tg.nn.SGConv(in_c, hid_c, K=K, cached=False, bias=True)
    elif conv_type == 'tag':
        return tg.nn.TAGConv(in_c, hid_c, K=K, bias=True)
    elif conv_type == 'arma':
        return tg.nn.ARMAConv(in_c, hid_c, num_stacks=1, num_layers=K, shared_weights=True, dropout=0.0, bias=True)
    elif conv_type == 'bern':
        return tg.nn.BernConv(in_c, hid_c, K=K, bias=True)
    else:
        raise ValueError(f"conv_type must be one of {CONV_TYPES}, got: {conv_type!r}")


def run_view_conv(conv, conv_type, x, edge_index, edge_weight):
    """Calls a view's conv module with whatever signature it actually needs,
    given the common (x, edge_index, edge_weight) triple built in forward().
    edge_weight is always the same tensor (static / learnable / learnable_scratch,
    depending on graph_mode) -- only how each conv layer type consumes it differs."""
    if conv_type in ('cheb', 'gcn', 'graph'):
        return conv(x, edge_index, edge_weight)
    elif conv_type in ('gat', 'gin'):
        return conv(x, edge_index, edge_attr=edge_weight.unsqueeze(-1))

    elif conv_type == 'sage':
        return conv(x, edge_index)
    elif conv_type == 'sgc':
        return conv(x, edge_index, edge_weight)

    elif conv_type == 'arma':
        return conv(x, edge_index, edge_weight)

    elif conv_type == 'tag':
        return conv(x, edge_index, edge_weight)

    elif conv_type == 'bern':
        return conv(x, edge_index, edge_weight)
    elif conv_type == 'sage':
        return conv(x, edge_index)
    else:
        raise ValueError(f"conv_type must be one of {CONV_TYPES}, got: {conv_type!r}")


LEARNABLE_GRAPH_MODES = ('learnable', 'learnable_scratch')
EDGE_SCRATCH_INIT_SCALE = 0.1  # random init for graph_mode='learnable_scratch': U(-0.1, 0.1)

EDGE_DEGREE_EPS = 1e-6  # keeps deg**-0.5 finite even if signed edge weights nearly
                        # cancel out at a node (see SafeChebConv below)


def safe_get_laplacian(edge_index, edge_weight=None, normalization=None,
                        dtype=None, num_nodes=None):
    """Drop-in replacement for torch_geometric.utils.get_laplacian.

    Root cause of the NaN blow-up with learnable edge weights: ChebConv's 'sym'
    normalization computes, for every node, deg = sum of its incident edge
    weights, then deg ** -0.5. Our edge weights are signed partial correlations,
    so once they become learnable a single optimizer step can push deg to zero or
    negative for some node -- and (negative)**-0.5 is NaN (torch_geometric's own
    get_laplacian only special-cases +inf, not NaN), which immediately poisons
    every parameter that shares the loss.

    Fix: compute the 1/sqrt(deg) (or 1/deg) normalization factor from the
    *absolute value* of deg, clamped away from zero. This keeps the normalization
    meaningful (it reflects each node's total connection *strength*, irrespective
    of sign) while the signed edge weight itself is still what gets propagated as
    the actual message, so the sign of each partial correlation is preserved.
    """
    edge_index, edge_weight = remove_self_loops(edge_index, edge_weight)
    if edge_weight is None:
        edge_weight = torch.ones(edge_index.size(1), dtype=dtype, device=edge_index.device)

    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    row, col = edge_index[0], edge_index[1]
    deg = pyg_scatter(edge_weight, row, 0, dim_size=num_nodes, reduce='sum')
    deg_safe = deg.abs().clamp_min(EDGE_DEGREE_EPS)

    if normalization is None:
        # L = D - A.
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        edge_weight = torch.cat([-edge_weight, deg_safe], dim=0)
    elif normalization == 'sym':
        # A_norm = -D^{-1/2} A D^{-1/2}.
        deg_inv_sqrt = deg_safe.pow(-0.5)
        edge_weight = deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]
        # L = I - A_norm.
        edge_index, edge_weight = add_self_loops(edge_index, -edge_weight, fill_value=1., num_nodes=num_nodes)
    else:  # 'rw'
        # A_norm = -D^{-1} A.
        deg_inv = 1.0 / deg_safe
        edge_weight = deg_inv[row] * edge_weight
        # L = I - A_norm.
        edge_index, edge_weight = add_self_loops(edge_index, -edge_weight, fill_value=1., num_nodes=num_nodes)

    return edge_index, edge_weight


class SafeChebConv(tg.nn.ChebConv):
    """ChebConv whose graph-Laplacian normalization can never produce NaN, even
    when edge_weight is signed and learnable. Identical to tg.nn.ChebConv in
    every other respect (same forward pass, same parameters, same behavior for
    non-negative / static edge weights); only __norm__ is overridden to route
    through safe_get_laplacian instead of the stock get_laplacian."""

    def __norm__(self, edge_index, num_nodes, edge_weight, normalization,
                 lambda_max=None, dtype=None, batch=None):
        edge_index, edge_weight = safe_get_laplacian(
            edge_index, edge_weight, normalization, dtype, num_nodes)
        assert edge_weight is not None

        if lambda_max is None:
            lambda_max = 2.0 * edge_weight.max()
        elif not isinstance(lambda_max, torch.Tensor):
            lambda_max = torch.tensor(lambda_max, dtype=dtype, device=edge_index.device)
        assert lambda_max is not None

        if batch is not None and lambda_max.numel() > 1:
            lambda_max = lambda_max[batch[edge_index[0]]]

        edge_weight = (2.0 * edge_weight) / lambda_max
        edge_weight.masked_fill_(edge_weight == float('inf'), 0)

        loop_mask = edge_index[0] == edge_index[1]
        edge_weight[loop_mask] -= 1

        return edge_index, edge_weight


############################################### View-fusion method (fusion_type) ###################################################
############################################### View-fusion method (fusion_type) ###################################################
############################################### View-fusion method (fusion_type) ###################################################
# How the per-view embeddings (one (n_subjects, hid_c) vector per view, after
# global_mean_pool) get combined into the single vector that is fed to the final
# linear classifier. Configured from the notebook via MultiViewGCN(...,
# fusion_type=...) -- see the FUSION_TYPE cell below (next to CONV_TYPE) to change it.
#
#   'concat'    : THE ORIGINAL / DEFAULT PATH, byte-for-byte unchanged -- the
#                 n_views embeddings are concatenated (torch.cat) into one
#                 hid_c * n_views vector and the classifier is nn.Linear(hid_c *
#                 n_views, out_c), exactly as before. Leaving fusion_type='concat'
#                 reproduces exactly the results you were already getting;
#                 nothing else about the model changes.
#   'sum'       : element-wise sum of the n_views embeddings -> one hid_c vector.
#                 No extra parameters; simplest alternative to concat.
#   'mean'      : element-wise mean of the n_views embeddings -> one hid_c vector.
#                 Same as 'sum' but scale-normalized by n_views.
#   'attention' : a small shared nn.Linear(hid_c, 1) scores each view's embedding
#                 per subject; scores are softmax-normalized across views and
#                 used as attention weights for a weighted sum -> one hid_c
#                 vector. Lets the model learn "how much to trust" each view
#                 (aseg/aparc/wmparc) per subject, instead of always using a
#                 fixed hid_c-per-view split like concat does.
#   'gated'     : a learned gate (nn.Linear(hid_c * n_views, hid_c * n_views) +
#                 sigmoid, i.e. a highway/GLU-style gate) computed from the
#                 concatenation of all views assigns a per-dimension (not just
#                 per-view) weight to every view's embedding before they are
#                 summed -> one hid_c vector. Strictly more expressive than
#                 'attention' (which only has one scalar weight per view),
#                 at the cost of extra parameters.
#
# In every non-'concat' mode the fused vector has size hid_c (not hid_c *
# n_views), so the classifier's input size is adjusted automatically inside
# MultiViewGCN.__init__ -- no other code needs to change.
FUSION_TYPES = ('concat', 'sum', 'mean', 'attention', 'gated')


def build_fusion_module(fusion_type, hid_c, n_views):
    """Builds whatever extra nn.Module a given fusion_type needs (or None, for
    the parameter-free fusion types). Called once, in MultiViewGCN.__init__."""
    if fusion_type in ('concat', 'sum', 'mean'):
        return None
    elif fusion_type == 'attention':
        return nn.Linear(hid_c, 1)
    elif fusion_type == 'gated':
        return nn.Linear(hid_c * n_views, hid_c * n_views)
    else:
        raise ValueError(f"fusion_type must be one of {FUSION_TYPES}, got: {fusion_type!r}")


def fuse_view_embeddings(fusion_type, fusion_module, view_embeddings):
    """Combines the per-view embeddings into the single vector fed to the
    classifier.

    view_embeddings : list[Tensor], n_views tensors of shape (n_subjects, hid_c).

    Returns a tensor of shape (n_subjects, hid_c * n_views) for fusion_type=
    'concat' (unchanged from before), or (n_subjects, hid_c) for every other
    fusion_type.
    """
    if fusion_type == 'concat':
        # Original behavior -- untouched.
        return torch.cat(view_embeddings, dim=1)

    stacked = torch.stack(view_embeddings, dim=1)  # (n_subjects, n_views, hid_c)

    if fusion_type == 'sum':
        return stacked.sum(dim=1)
    elif fusion_type == 'mean':
        return stacked.mean(dim=1)
    elif fusion_type == 'attention':
        scores = fusion_module(stacked).squeeze(-1)            # (n_subjects, n_views)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)   # (n_subjects, n_views, 1)
        return (weights * stacked).sum(dim=1)                  # (n_subjects, hid_c)
    elif fusion_type == 'gated':
        n_subjects, n_views, hid_c = stacked.shape
        concat_all = stacked.reshape(n_subjects, n_views * hid_c)
        gates = torch.sigmoid(fusion_module(concat_all))       # (n_subjects, n_views * hid_c)
        gated = (concat_all * gates).reshape(n_subjects, n_views, hid_c)
        return gated.sum(dim=1)                                # (n_subjects, hid_c)
    else:
        raise ValueError(f"fusion_type must be one of {FUSION_TYPES}, got: {fusion_type!r}")


class MultiViewGCN(nn.Module):
    def __init__(self, in_channels_list, hid_c, out_c, K, dropout_rate,
                 graph_mode='static', base_edge_weights=None, n_subjects=None,
                 conv_type='cheb', fusion_type='concat'):
        """
        :param in_channels_list: list[int], number of sub-features per node, for each view.
        :param hid_c: int, hidden size used by every view branch.
        :param out_c: int, number of output classes.
        :param K: Chebyshev filter size.
        :param dropout_rate: float, dropout applied after every view branch and before the classifier.
        :param graph_mode: 'static', 'learnable', or 'learnable_scratch' (see note above).
        :param conv_type: 'cheb' (default, original ChebConv path), 'gcn', 'graph',
            'gat', 'gin', or 'sage' -- which message-passing layer each view branch
            uses (see the CONV_TYPES note above build_view_conv). Default 'cheb'
            reproduces the exact original results.
        :param fusion_type: 'concat' (default, original torch.cat + Linear(hid_c *
            n_views, out_c) path), 'sum', 'mean', 'attention', or 'gated' -- how the
            per-view embeddings are combined before the final classifier (see the
            FUSION_TYPES note above build_fusion_module). Default 'concat'
            reproduces the exact original results.
        :param base_edge_weights: list[np.ndarray], per view -- the *single-copy*
            edge weights (before tiling across subjects). For graph_mode='learnable'
            these values are used to initialize the learnable parameter. For
            graph_mode='learnable_scratch' only their *shape* (i.e. the number of
            edges per view -- the topology) is used; the actual initial values are
            drawn randomly instead. Required for both learnable modes.
        :param n_subjects: number of subjects in this fold (needed to repeat the
            learnable weights over the block-diagonal batched graph, exactly like
            tile_graph_for_batch). Only needed for learnable modes.
        """
        super(MultiViewGCN, self).__init__()
        self.n_views = len(in_channels_list)
        self.graph_mode = graph_mode
        self.n_subjects = n_subjects
        self.conv_type = conv_type
        if self.conv_type not in CONV_TYPES:
            raise ValueError(f"conv_type must be one of {CONV_TYPES}, got: {conv_type!r}")

        self.fusion_type = fusion_type
        if self.fusion_type not in FUSION_TYPES:
            raise ValueError(f"fusion_type must be one of {FUSION_TYPES}, got: {fusion_type!r}")

        self.view_convs = nn.ModuleList([
            build_view_conv(conv_type, in_c, hid_c, K)
            for in_c in in_channels_list
        ])
        # One independent ReLU module per view (NOT a single shared instance).
        # DeepLift/DeepLiftShap register per-call hooks on each nonlinear
        # activation module and require every such module to be used exactly
        # once per forward pass -- a single self.relu called once per view (as
        # it was before) violates that and captum raises
        # "module ... does not contain some of the input/output attributes
        # required for DeepLift computations". Using inplace=False too, since
        # captum's own docs warn in-place activations can corrupt the saved
        # activations DeepLift's hooks rely on. Purely a wiring fix: ReLU has no
        # parameters and is elementwise, so this changes neither the forward
        # computation nor training results -- only removes the hook conflict.
        self.relus = nn.ModuleList([nn.ReLU(inplace=False) for _ in range(self.n_views)])
        self.dropout = nn.Dropout(dropout_rate)
        self.fusion_module = build_fusion_module(fusion_type, hid_c, self.n_views)
        # 'concat' keeps the original classifier input size (hid_c * n_views);
        # every other fusion_type first combines the views down to one hid_c
        # vector, so the classifier only needs hid_c inputs.
        classifier_in = hid_c * self.n_views if fusion_type == 'concat' else hid_c
        self.classifier = nn.Linear(classifier_in, out_c)
        self.softmax = nn.LogSoftmax(dim=1)

        if self.graph_mode in LEARNABLE_GRAPH_MODES:
            if base_edge_weights is None or n_subjects is None:
                raise ValueError(
                    f"graph_mode={graph_mode!r} requires base_edge_weights and n_subjects "
                    "(base_edge_weights supplies the edge topology either way)."
                )
            if self.graph_mode == 'learnable':
                # Initial value = the current partial-correlation weights: training
                # starts equivalent to the static graph and only fine-tunes it.
                self.edge_weight_params = nn.ParameterList([
                    nn.Parameter(torch.as_tensor(w, dtype=torch.float32).clone())
                    for w in base_edge_weights
                ])
            else:  # 'learnable_scratch'
                # Same number of edges per view (same topology) as base_edge_weights,
                # but the *values* are not taken from the partial-correlation prior --
                # each weight starts at a small random value and is learned purely
                # from the training signal.
                self.edge_weight_params = nn.ParameterList([
                    nn.Parameter(
                        torch.empty(len(w), dtype=torch.float32).uniform_(
                            -EDGE_SCRATCH_INIT_SCALE, EDGE_SCRATCH_INIT_SCALE)
                    )
                    for w in base_edge_weights
                ])
        elif self.graph_mode != 'static':
            raise ValueError(
                f"graph_mode must be 'static', 'learnable', or 'learnable_scratch', got: {graph_mode!r}"
            )

    def forward(self, data_list, edge_index_list, edgenet_input_list, batch_vec_list):
        view_embeddings = []
        for i in range(self.n_views):
            if self.graph_mode in LEARNABLE_GRAPH_MODES:
                # Same edges (edge_index stays fixed), only the weights are
                # learnable, broadcast over every subject (whose block-diagonal
                # structure repeats identically in the tiled graph).
                edge_weight = self.edge_weight_params[i].repeat(self.n_subjects)
            else:
                edge_weight = torch.squeeze(edgenet_input_list[i])
            h = run_view_conv(self.view_convs[i], self.conv_type, data_list[i], edge_index_list[i], edge_weight)
            h = self.relus[i](h)
            h = self.dropout(h)
            h = tg.nn.global_mean_pool(h, batch_vec_list[i])   # (n_subjects, hid_c)
            view_embeddings.append(h)

        h = fuse_view_embeddings(self.fusion_type, self.fusion_module, view_embeddings)
        h = self.dropout(h)
        out = self.classifier(h)

        return self.softmax(out)


In [11]:
############################################### GCN labels ###############################################################
############################################### GCN labels ###############################################################
############################################### GCN labels ###############################################################
# One-hot labels for the GCN's 2-class softmax output.
GCN_labels = np.zeros((number_samples, 2))
for i in range(len(labels)):
    if labels[i] == 1:
        GCN_labels[i, 0] = 1
    else:
        GCN_labels[i, 1] = 1


In [12]:
# def distance_correlation_matrix(node_feats):
#     """
#     Compute pairwise distance correlation between ROI/node columns.

#     Parameters
#     ----------
#     node_feats : np.ndarray
#         Shape: (n_subjects, n_nodes)

#     Returns
#     -------
#     sim : np.ndarray
#         Shape: (n_nodes, n_nodes)
#         Values in [0, 1].
#     """
#     X = np.asarray(node_feats, dtype=np.float64)

#     n_subjects, n_nodes = X.shape
#     sim = np.zeros((n_nodes, n_nodes), dtype=np.float64)

#     for i in range(n_nodes):
#         xi = X[:, i]

#         # Centered distance matrix for ROI i
#         a = np.abs(xi[:, None] - xi[None, :])
#         a_mean_row = a.mean(axis=1, keepdims=True)
#         a_mean_col = a.mean(axis=0, keepdims=True)
#         a_mean_all = a.mean()

#         A = a - a_mean_row - a_mean_col + a_mean_all

#         for j in range(i, n_nodes):
#             xj = X[:, j]

#             b = np.abs(xj[:, None] - xj[None, :])
#             b_mean_row = b.mean(axis=1, keepdims=True)
#             b_mean_col = b.mean(axis=0, keepdims=True)
#             b_mean_all = b.mean()

#             B = b - b_mean_row - b_mean_col + b_mean_all

#             dcov2 = np.mean(A * B)
#             dvar_x = np.mean(A * A)
#             dvar_y = np.mean(B * B)

#             denominator = np.sqrt(dvar_x * dvar_y)

#             if denominator > 1e-12:
#                 dcor = np.sqrt(max(dcov2, 0.0) / denominator)
#             else:
#                 dcor = 0.0

#             sim[i, j] = dcor
#             sim[j, i] = dcor

#     np.fill_diagonal(sim, 1.0)

#     return sim


def distance_correlation_matrix(node_feats):
    """
    Fast vectorized pairwise Distance Correlation.

    Parameters
    ----------
    node_feats : np.ndarray
        Shape: (n_subjects, n_nodes)

    Returns
    -------
    sim : np.ndarray
        Shape: (n_nodes, n_nodes)
        Distance correlation matrix, values approximately in [0, 1].
    """

    X = np.asarray(node_feats, dtype=np.float64)

    n_subjects, n_nodes = X.shape

    # ---------------------------------------------------------
    # 1. Pairwise absolute distances for ALL nodes at once
    #
    # D[s1, s2, node] =
    #     |X[s1, node] - X[s2, node]|
    #
    # Shape: (n_subjects, n_subjects, n_nodes)
    # ---------------------------------------------------------
    D = np.abs(
        X[:, None, :] - X[None, :, :]
    )

    # ---------------------------------------------------------
    # 2. Double-center each distance matrix
    # ---------------------------------------------------------
    row_mean = D.mean(axis=1, keepdims=True)
    col_mean = D.mean(axis=0, keepdims=True)
    total_mean = D.mean(axis=(0, 1), keepdims=True)

    A = D - row_mean - col_mean + total_mean

    # We no longer need D
    del D

    # ---------------------------------------------------------
    # 3. Distance covariance:
    #
    # dcov^2(i,j) = mean(A_i * A_j)
    #
    # We can calculate ALL ROI pairs with one matrix
    # multiplication instead of a nested loop.
    # ---------------------------------------------------------
    A_flat = A.reshape(n_subjects * n_subjects, n_nodes)

    dcov2 = (
        A_flat.T @ A_flat
    ) / (n_subjects * n_subjects)

    del A
    del A_flat

    # ---------------------------------------------------------
    # 4. Distance variances
    # ---------------------------------------------------------
    dvar = np.diag(dcov2)

    # Numerical safety
    dvar = np.maximum(dvar, 0.0)

    # ---------------------------------------------------------
    # 5. Distance correlation
    #
    # dCor(i,j) =
    # sqrt(
    #     dcov^2(i,j) /
    #     sqrt(dvar(i) * dvar(j))
    # )
    # ---------------------------------------------------------
    denominator = np.sqrt(
        np.outer(dvar, dvar)
    )

    sim = np.zeros_like(dcov2)

    valid = denominator > 1e-12

    sim[valid] = np.sqrt(
        np.maximum(
            dcov2[valid] / denominator[valid],
            0.0
        )
    )

    # Constant features -> no dependency
    sim[~np.isfinite(sim)] = 0.0

    # Self similarity
    np.fill_diagonal(sim, 1.0)

    return sim
from sklearn.feature_selection import mutual_info_regression


def mutual_information_matrix(node_feats):
    """
    Compute pairwise Mutual Information between ROI/node columns.

    Parameters
    ----------
    node_feats : np.ndarray
        Shape: (n_subjects, n_nodes)

    Returns
    -------
    sim : np.ndarray
        Shape: (n_nodes, n_nodes)
        Symmetric normalized MI matrix in [0, 1].
    """

    X = np.asarray(node_feats, dtype=np.float64)

    n_subjects, n_nodes = X.shape

    sim = np.zeros(
        (n_nodes, n_nodes),
        dtype=np.float64
    )

    # ---------------------------------------------------------
    # Compute only the upper triangular part.
    #
    # MI(X,Y) = MI(Y,X), so each pair only needs to be
    # estimated ONCE.
    # ---------------------------------------------------------
    for i in range(n_nodes):

        xi = X[:, i]

        for j in range(i + 1, n_nodes):

            xj = X[:, j]

            # If either feature is constant, there is no
            # meaningful dependency to estimate.
            if np.std(xi) < 1e-12 or np.std(xj) < 1e-12:
                mi = 0.0

            else:
                mi = mutual_info_regression(
                    xi.reshape(-1, 1),
                    xj,
                    random_state=42
                )[0]

                # Numerical safety
                if not np.isfinite(mi):
                    mi = 0.0

                mi = max(float(mi), 0.0)

            # MI is symmetric
            sim[i, j] = mi
            sim[j, i] = mi

    # ---------------------------------------------------------
    # Normalize MI to [0, 1]
    # ---------------------------------------------------------
    max_mi = np.max(sim)

    if max_mi > 0:
        sim /= max_mi

    # Self-similarity
    np.fill_diagonal(sim, 1.0)

    return sim


def mutual_information_matrix1(node_feats):
    """
    Compute pairwise mutual information between ROI/node columns.

    Parameters
    ----------
    node_feats : np.ndarray
        Shape: (n_subjects, n_nodes)

    Returns
    -------
    sim : np.ndarray
        Shape: (n_nodes, n_nodes)
    """
    X = np.asarray(node_feats, dtype=np.float64)

    n_subjects, n_nodes = X.shape
    sim = np.zeros((n_nodes, n_nodes), dtype=np.float64)

    for i in range(n_nodes):
        xi = X[:, i]

        for j in range(i, n_nodes):
            xj = X[:, j]

            # MI(X_i, X_j)
            mi_ij = mutual_info_regression(
                xi.reshape(-1, 1),
                xj,
                random_state=42
            )[0]

            # MI is theoretically symmetric, but the estimator can have
            # small numerical/asymmetry differences depending on direction.
            mi_ji = mutual_info_regression(
                xj.reshape(-1, 1),
                xi,
                random_state=42
            )[0]

            mi = 0.5 * (mi_ij + mi_ji)

            sim[i, j] = mi
            sim[j, i] = mi

    np.fill_diagonal(sim, 1.0)

    return sim

In [13]:
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.covariance import LedoitWolf


# Per-view k (number of nearest-neighbor edges per ROI node). Each view has its
# own k instead of sharing one global value. The values below (32/32/32) match the
# old single K_NEIGHBORS=32 that used to apply to every view, so as long as you
# leave them at 32 the graphs -- and therefore the results -- are IDENTICAL to
# before. Change any entry to give that view a different k.
K_NEIGHBORS_PER_VIEW = {
    'aseg':   4,
    'aparc':  16,
    'wmparc': 8,
}


DEFAULT_K_NEIGHBORS = 32   # only used as a fallback if a view is missing from K_NEIGHBORS_PER_VIEW above
                           # (self-contained default -- does NOT depend on any other variable,
                           # so it can never break even if you edit/remove things above)

GRAPH_METRIC = 'pearson'   # 'pearson', 'spearman', 'cosine', 'euclidean', 'partial_correlation'

def compute_similarity(node_feats, metric):
    if metric == 'pearson':
        sim = np.corrcoef(node_feats.T)

    elif metric == 'spearman':
        # # sim, _ = spearmanr(node_feats)
        # # if node_feats.shape[1] == 2:
        # #     sim = np.array([[1.0, sim], [sim, 1.0]])

        # sim, _ = spearmanr(node_feats)
        # # spearmanr returns a single SCALAR (not an (n_nodes, n_nodes) matrix)
        # # in two situations: (1) there are exactly 2 node columns, or (2) EVERY
        # # column is constant (e.g. this view's representative sub-feature --
        # # index 0 -- wasn't kept by Ridge/RFE for any node here, so every
        # # column is all-zero). Case (2) is what was crashing fill_diagonal()
        # # below with 'array must be at least 2-d'. Reshape either case back
        # # into a proper matrix so the rest of this function keeps working.
        # sim = np.asarray(sim)
        # if sim.ndim == 0:
        #     n_nodes = node_feats.shape[1]
        #     rho = float(sim)
        #     sim = np.full((n_nodes, n_nodes), rho)
        #     np.fill_diagonal(sim, 1.0)


        sim, _ = spearmanr(node_feats)
        sim = np.asarray(sim, dtype=float)

        if sim.ndim == 0:
            n_nodes = node_feats.shape[1]
            sim = np.zeros((n_nodes, n_nodes), dtype=float)
            np.fill_diagonal(sim, 1.0)

    elif metric == 'cosine':
        sim = cosine_similarity(node_feats.T)

    elif metric == 'euclidean':
        dist = squareform(pdist(node_feats.T, metric='euclidean'))
        sigma = np.median(dist[dist > 0])
        sim = np.exp(-(dist ** 2) / (2 * sigma ** 2))

    elif metric == 'partial_correlation':
        lw = LedoitWolf()
        lw.fit(node_feats)
        precision = lw.precision_
        d = np.sqrt(np.diag(precision))
        d[d == 0] = 1e-8   # avoid division by zero for fully-constant columns
        sim = -precision / np.outer(d, d)

    elif metric == 'distance_correlation':
        sim = distance_correlation_matrix(node_feats)

    elif metric == 'mutual_information':
        sim = mutual_information_matrix(node_feats)
    else:
        raise ValueError(f'Unknown metric: {metric}')

    return np.nan_to_num(sim, nan=0.0)


def get_k_for_view(view):
    """Looks up this view's k in K_NEIGHBORS_PER_VIEW, falling back to
    DEFAULT_K_NEIGHBORS if the view isn't listed there."""
    return K_NEIGHBORS_PER_VIEW.get(view, DEFAULT_K_NEIGHBORS)


def build_covariance_graph(view, train_positions, k=None, metric=GRAPH_METRIC):
    """
    Structural covariance graph: correlates one representative sub-feature of each
    ROI (its first sub-feature, e.g. NVoxels / NumVert) across the fold's training
    subjects only, to avoid leaking val/test subjects into the graph structure.
    Returns a kNN graph over the ROI nodes of that view.

    k: number of nearest-neighbor edges per node for THIS view. If not given
    explicitly, it's looked up per-view via get_k_for_view(view), so different
    views can use different k without changing any call site that doesn't care.
    """
    if k is None:
        k = get_k_for_view(view)
    node_feats = view_node_features[view][train_positions, :, 0]   # (n_train_subjects, n_nodes)
    sim = compute_similarity(node_feats, metric)
    np.fill_diagonal(sim, -np.inf)

    n_nodes = sim.shape[0]
    edge_list, weight_list = [], []
    for i in range(n_nodes):
        neighbors = np.argsort(sim[i])[-k:]
        for j in neighbors:
            edge_list.append([i, j]); weight_list.append(sim[i, j])
            edge_list.append([j, i]); weight_list.append(sim[i, j])

    edge_index = np.array(edge_list, dtype=np.int64).T
    edge_weight = np.array(weight_list, dtype=np.float32)
    return edge_index, edge_weight


def tile_graph_for_batch(edge_index, edge_weight, n_nodes, n_subjects):
    """Repeats the one shared brain-region graph once per subject (block-diagonal),
    so ChebConv can process every subject's graph in a single call over the
    (n_subjects * n_nodes) stacked node-feature matrix."""
    offsets = (np.arange(n_subjects) * n_nodes).reshape(-1, 1, 1)
    tiled = edge_index[None, :, :] + offsets                 # (n_subjects, 2, n_edges)
    edge_index_b = tiled.transpose(1, 0, 2).reshape(2, -1).astype(np.int64)
    edge_weight_b = np.tile(edge_weight, n_subjects).reshape(-1, 1).astype(np.float32)
    return edge_index_b, edge_weight_b


data = {}
# base_edge_weights[fold] = list[np.ndarray], per view -- the *single-copy*
# edge weights (before tiling across subjects). Only needed to initialize the
# learnable graph parameter; unused in 'static' mode, so it doesn't affect current behavior.
base_edge_weights = {}
for fold in range(1, k_fold + 1):
    data[str(fold)] = [[], GCN_labels, [], []]
    base_edge_weights[str(fold)] = []

    for view in view_names:
        n_nodes = view_node_features[view].shape[1]
        k_view = get_k_for_view(view)   # <-- per-view k instead of one shared K_NEIGHBORS
        edge_index, edge_weight = build_covariance_graph(view, dist_train[str(fold)], k=k_view)
        edge_index_b, edge_weight_b = tile_graph_for_batch(edge_index, edge_weight, n_nodes, number_samples)

        data[str(fold)][0].append(X_BATCHED[view])
        data[str(fold)][2].append(edge_index_b)
        data[str(fold)][3].append(edge_weight_b)
        base_edge_weights[str(fold)].append(edge_weight)


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [14]:
############################################### Train, validaiton, test pipelines ###############################################################
############################################### Train, validaiton, test pipelines ###############################################################
############################################### Train, validaiton, test pipelines ###############################################################
BATCH_VEC_TENSORS = None   # built once, lazily, as soon as we know if we're on cuda

def _get_batch_vec_tensors(args):
    global BATCH_VEC_TENSORS
    if BATCH_VEC_TENSORS is None:
        tensors = [torch.from_numpy(BATCH_VEC[v]) for v in view_names]
        if args.cuda:
            tensors = [t.cuda() for t in tensors]
        BATCH_VEC_TENSORS = tensors
    return BATCH_VEC_TENSORS


def _prepare_multiview_batch(all_data, args, requires_grad=False):
    data_list = [torch.from_numpy(d).float() for d in all_data[0]]
    target = torch.from_numpy(all_data[1]).float().long()
    edge_index_list = [torch.from_numpy(ei) for ei in all_data[2]]
    edgenet_input_list = [torch.from_numpy(ew).float() for ew in all_data[3]]

    if requires_grad:
        for d in data_list:
            d.requires_grad_(True)
        for ew in edgenet_input_list:
            ew.requires_grad_(True)

    if args.cuda:
        data_list = [d.cuda() for d in data_list]
        target = target.cuda()
        edge_index_list = [ei.cuda() for ei in edge_index_list]
        edgenet_input_list = [ew.cuda() for ew in edgenet_input_list]

    batch_vec_list = _get_batch_vec_tensors(args)
    return data_list, target, edge_index_list, edgenet_input_list, batch_vec_list


def train_GCN(args, model, all_data, fold_train_index, fold_validation_index, fold_test_index, scheduler):

    model.train()
    data_list, target, edge_index_list, edgenet_input_list, batch_vec_list = _prepare_multiview_batch(all_data, args, requires_grad=True)

    scheduler.zero_grad()
    out = model(data_list, edge_index_list, edgenet_input_list, batch_vec_list)
    out = out[fold_train_index, :]
    train_target = target[fold_train_index, :]

    cross_loss = torch.nn.functional.nll_loss(out, torch.max(train_target, 1)[1])  # only train set will be included
    out = torch.max(out, 1)[1]
    train_target = torch.max(train_target, 1)[1]
    train_target = train_target.cpu().numpy()
    out = out.cpu().numpy()
    cross_loss.backward()

    # --- Guard against exploding/NaN gradients (mostly relevant for graph_mode='learnable') ---
    # ChebConv's 'sym' normalization can produce a very large or NaN gradient once the
    # sum of a node's edge weights gets close to zero. First check that the gradients are
    # finite; if not, skip this optimizer step entirely (leave parameters untouched) so
    # Adam's internal state doesn't get corrupted. Then clip the gradient so a single bad
    # step can't blow up the parameters.
    grads_ok = all(
        p.grad is None or torch.isfinite(p.grad).all()
        for p in model.parameters()
    )
    if grads_ok:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.grad_clip_norm)
        scheduler.step()
    else:
        print('  [warning] NaN/Inf gradient detected -- this step was skipped (parameters not updated).')
        scheduler.zero_grad()

    # Keep the learnable edge weights inside the same physical partial-correlation range
    # (-1..1), so the 'sym' normalization never encounters extreme magnitudes.
    real_model = model.module if isinstance(model, nn.parallel.DataParallel) else model
    if getattr(real_model, 'graph_mode', 'static') in LEARNABLE_GRAPH_MODES:
        with torch.no_grad():
            for p in real_model.edge_weight_params:
                p.clamp_(-1.0, 1.0)

    ############################## validation and test ################################################################################
    val_target, val_out, val_out_prob, val_loss = validation_GCN(args, model, all_data, fold_validation_index)
    test_target, test_out, test_out_prob, test_loss = test_GCN(args, model, all_data, fold_test_index)

    return model, train_target, out, cross_loss.item(), val_target, val_out, val_out_prob, val_loss, test_target, test_out, test_out_prob, test_loss


def validation_GCN(args, model, all_data, fold_validation_index):

    model.eval()
    data_list, target, edge_index_list, edgenet_input_list, batch_vec_list = _prepare_multiview_batch(all_data, args, requires_grad=False)

    out = model(data_list, edge_index_list, edgenet_input_list, batch_vec_list)
    out = out[fold_validation_index, :]
    target = target[fold_validation_index, :]
    cross_loss = torch.nn.functional.nll_loss(out, torch.max(target, 1)[1])

    out_prob = torch.exp(out)   # log-softmax -> probabilities, same as test_GCN below
    out = torch.max(out, 1)[1]
    target = torch.max(target, 1)[1]
    target = target.cpu().numpy()
    out = out.cpu().numpy()
    out_prob = out_prob.cpu().detach().numpy()

    return target, out, out_prob, cross_loss.item()


def test_GCN(args, model, all_data, fold_test_index):

    model.eval()
    data_list, target, edge_index_list, edgenet_input_list, batch_vec_list = _prepare_multiview_batch(all_data, args, requires_grad=False)

    out = model(data_list, edge_index_list, edgenet_input_list, batch_vec_list)
    out = out[fold_test_index, :]
    target = target[fold_test_index, :]
    cross_loss = torch.nn.functional.nll_loss(out, torch.max(target, 1)[1])

    out_prob = torch.exp(out)
    out = torch.max(out, 1)[1]
    target = torch.max(target, 1)[1]
    target = target.cpu().numpy()
    out = out.cpu().numpy()
    out_prob = out_prob.cpu().detach().numpy()

    return target, out, out_prob, cross_loss.item()


In [15]:
########################################### Training setting ################################################
########################################### Training setting ################################################
########################################### Training setting ################################################

import random

parser = argparse.ArgumentParser()
parser.add_argument('--ngpu', type=int, default=1)
parser.add_argument('--nEpochs', type=int, default=100)
parser.add_argument('--weight-decay', '--wd', default=1e-8, type=float,
                    metavar='W', help='weight decay (default: 5e-4, raisd from the single-view notebook to help fight overfitting)')
parser.add_argument('--no-cuda', action='store_true')
parser.add_argument('--lr', type=str, default=1e-3)
parser.add_argument('--seed', type=int, default=1)
parser.add_argument('--cheby_order_K', type=int, default=5)
parser.add_argument('--hidden_dimension', type=int, default=128)
parser.add_argument('--output_dimension', type=int, default=2)
parser.add_argument('--dropout_rate', type=int, default=0.5)
parser.add_argument('--grad_clip_norm', type=float, default=5.0,
                    help='max grad-norm for clipping (important for graph_mode=learnable, which can be unstable)')
parser.add_argument('--edge_lr_scale', type=float, default=0.3,
                    help='learning-rate scale factor applied to the learnable edge weights (relative to args.lr)')

args, unknown = parser.parse_known_args()
args.cuda = not args.no_cuda and torch.cuda.is_available()
torch.manual_seed(args.seed)
if args.cuda:
    torch.cuda.manual_seed(args.seed)

gpu_ids = range(args.ngpu)
train = train_GCN


def set_full_seed(seed):
    """Resets EVERY relevant RNG. Called once per fold -- both here in the main
    training loop below AND later in the ROAR/explainability cells -- so every
    fold starts from an identical, reproducible RNG state. This is what makes
    RECOMPUTE_BASELINE / ROAR threshold=0 reproduce these exact results later."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # cudnn.deterministic gets you very close to bit-exact on GPU for the conv/
    # linear ops here; it does NOT fully cover every torch_geometric scatter
    # kernel, so for a strict bit-for-bit check run once on CPU.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Best-effort full determinism: forces PyTorch's own ops onto deterministic
    # kernels (errors instead of silently using a nondeterministic one where no
    # deterministic kernel exists, e.g. some torch_geometric scatter kernels --
    # hence warn_only=True: warn and fall back rather than crash training).
    torch.use_deterministic_algorithms(True, warn_only=True)



In [16]:
########################################### Graph mode (static vs learnable) ################################################
########################################### Graph mode (static vs learnable) ################################################
########################################### Graph mode (static vs learnable) ################################################
# 'static'            : the same fixed partial-correlation graph built in the previous
#                        cell, unchanged throughout training -- this mode should
#                        reproduce the exact same results as before.
# 'learnable'          : the weights of that same graph (topology/structure stays
#                        identical, only the weights change) are defined as learnable
#                        parameters and updated by the optimizer along with the rest of
#                        the model. They are *initialized* to exactly the current
#                        partial-correlation values, so training starts out equivalent
#                        to 'static' and only fine-tunes that prior.
# 'learnable_scratch'  : same topology as above (which regions are connected doesn't
#                        change), but the edge weights are no longer initialized from
#                        the partial-correlation prior -- they start at a small random
#                        value and are learned purely from the training signal, so the
#                        model isn't handed any pre-computed correlation strength.
GRAPH_MODE = 'static'   # 'static', 'learnable', or 'learnable_scratch'


In [17]:
########################################### Message-passing layer (conv_type) ################################################
########################################### Message-passing layer (conv_type) ################################################
########################################### Message-passing layer (conv_type) ################################################
# Which GNN layer each view branch uses internally (full per-type notes are in
# build_view_conv, in the model cell above).
#   'cheb'   : Chebyshev spectral conv (SafeChebConv) -- THE ORIGINAL DEFAULT.
#              Leaving CONV_TYPE = 'cheb' reproduces exactly the same results
#              you were already getting; nothing else changes.
#   'gcn'    : standard GCN (Kipf & Welling), edge_weight used directly.
#   'graph'  : GraphConv -- weighted sum, no Laplacian normalization (safest
#              simple choice to combine with learnable/signed edge weights).
#   'gat'    : Graph Attention -- edge weight fed in as a 1-dim edge feature.
#   'gin'    : GIN (via GINEConv) -- edge weight fed in as a 1-dim edge feature.
#   'sage'   : GraphSAGE -- ignores edge weights entirely, topology only.
CONV_TYPE = 'cheb'   # 'cheb', 'gcn', 'graph', 'gat', 'gin', or 'sage'


########################################### Fusion method (fusion_type) ################################################
########################################### Fusion method (fusion_type) ################################################
########################################### Fusion method (fusion_type) ################################################
# How the 3 view embeddings (aseg / aparc / wmparc) are combined before the final
# classifier. Full per-type notes are in the "View-fusion method" section of the
# model cell above (next to CONV_TYPES).
#   'concat'    : THE ORIGINAL / DEFAULT -- concatenate the 3 embeddings.
#                 Leaving FUSION_TYPE = 'concat' reproduces exactly the same
#                 results you were already getting; nothing else changes.
#   'sum'       : element-wise sum of the 3 embeddings (no extra parameters).
#   'mean'      : element-wise mean of the 3 embeddings (no extra parameters).
#   'attention' : learned per-view, per-subject attention weights (softmax over
#                 the 3 views), then a weighted sum.
#   'gated'     : learned per-dimension gate (sigmoid) computed from all 3 views
#                 together, then a gated sum -- more expressive than 'attention'.
FUSION_TYPE = 'concat'   # 'concat', 'sum', 'mean', 'attention', or 'gated'


# Explainability-only notebook (loads saved weights -- no retraining)

This notebook runs exactly the same data-loading / graph-building / model-definition
pipeline as the main notebook (the cells above, unchanged), but:

- **The main training loop (5-fold, hundreds of epochs) is removed** -- its output is not needed here.
- **Baseline retraining is also removed** -- instead, each fold's saved weights are read
  directly from `checkpoints/GCN_fold{fold}_best.pth` + `GCN_fold{fold}_recipe.json`
  (the same files the main notebook wrote via `save_best_model_and_recipe`).
- **The retraining sanity check (fold 1) is off by default** (a flag below turns it on if needed).
- From here on, only **explainability** (attribution methods + ROAR) runs -- ROAR itself
  does a short retrain per threshold; that is inherent to the ROAR method and can't be removed.

**To reproduce exactly the same results as before:**
1. The input files (`SMRI_FEATURES_CSV`, `PHENOTYPIC_CSV`) and `save_path` must be the
   same ones the checkpoints were built with.
2. Do not change any of the cells above (data loading / feature selection / graph
   building / model definition / training settings) -- the fold splits
   (`StratifiedKFold(random_state=0)`), graph construction, and model architecture
   must exactly match what was in effect when the weights were saved, or loading the
   state_dict will fail or the results will differ.
3. If there are no checkpoints on disk yet, run the full notebook once (with the
   training/baseline loop) to produce them; this notebook is only for later runs where
   you want explainability without retraining.


## Explainability (ROAR) — added on top of the existing training pipeline

This section implements the paper's idea (Vidya et al., eClinicalMedicine 2025)
on top of your actual `MultiViewGCN` model, not the paper's SSAE architecture
(your model is graph-based on sMRI, not a flat functional-connectivity vector).
What is kept faithful to the paper:

- **ROAR (Remove And Retrain)**: the most important features (per each
  interpretability method) are replaced with zero (since features are
  standardized, zero = "uninformative"), and the model is retrained **from
  scratch** with the same seed/hyperparameters; the accuracy drop at each
  threshold is the reliability measure for that interpretability method.
- **Removal thresholds**: the same list as the paper, `[0, 0.01, 0.05, 0.1, ..., 0.99]`.
- **7 interpretability methods**: Integrated Gradients, Guided Backprop,
  DeepLift, DeepLiftShap, GradientShap, LIME, KernelSHAP (the SHAP analogue —
  since your model is not tree-based, kernel/gradient SHAP is the correct
  substitute for a neural network).
- **Final brain-region ranking** based on the best method (steepest accuracy
  drop, exactly the paper's criterion), saved to CSV.

### The critical reproducibility point (as discussed)
For `threshold = 0.0` to reproduce the baseline exactly:
1. **Full seed reset** happens before every single retrain (not just once at
   the top of the notebook).
2. **Fold splits** (`dist_train/validation/test`) are reused exactly from the
   cells above — `StratifiedKFold` is never called again.
3. **Graph topology (edge_index/edge_weight) stays completely frozen** — ROAR
   only zeroes **node features** (`X_BATCHED`), it never rebuilds graph edges.
   If edges were rebuilt from the perturbed data too, `threshold=0` would no
   longer be guaranteed to match the baseline.
4. The original `X_BATCHED` is **never mutated in place**; every threshold
   starts from a fresh copy of the original array, not from the previous
   threshold's modified output (which would otherwise chain thresholds
   together).
5. For **bit-for-bit** reproducibility on GPU, `torch.backends.cudnn.deterministic=True`
   is set below; full bit-exactness on GPU is not guaranteed because of some
   `scatter` ops in torch_geometric — for a strict check, run once on CPU (the
   sanity-check cell below will show you the actual gap).

**Run order:** first run all the original cells above (0–18) so that
`data`, `dist_train/.../test`, `args`, `MultiViewGCN`, `X_BATCHED`,
`view_node_features`, `BATCH_VEC`, `base_edge_weights` all exist; the cells
below depend on those globals.


In [18]:
!pip install captum -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 32.0 MB/s eta 0:00:00


### 1) Reproducibility utilities + saving the best model's weights (+ full fold "recipe")

In [19]:
import random
import copy
import json
from pathlib import Path

XAI_DIR = os.path.join(save_path, "explainability")
CKPT_DIR = os.path.join(save_path, "checkpoints")
os.makedirs(XAI_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)


# set_full_seed(seed) is already defined in the "Training setting" cell above
# (cell that builds `args`) and is now also called per-fold inside the main
# training loop, so the baseline recompute below and threshold=0 in ROAR
# reproduce that main loop's numbers exactly. Not redefined here to avoid two
# copies drifting apart.


def build_fold_data(fold, x_override=None):
    """Rebuilds the (data_list, labels, edge_index_list, edgenet_input_list)
    tuple for a fold -- identical to data[str(fold)] when x_override is None.
    When x_override is given (a dict view -> np.ndarray with the SAME shape as
    X_BATCHED[view]), only the per-view NODE FEATURES are replaced; edge_index
    and edge_weight are ALWAYS reused unchanged from the original `data[str(fold)]`
    (see note above: never rebuild graph topology from perturbed features)."""
    orig = data[str(fold)]
    if x_override is None:
        data_list = orig[0]
    else:
        data_list = [x_override[v] for v in view_names]
    return [data_list, orig[1], orig[2], orig[3]]


def train_one_fold_from_scratch(fold, x_override=None, seed=None, nEpochs=None, verbose=True):
    """Mirrors the body of the main training-loop cell EXACTLY (same model
    construction, same optimizer param groups, same >= best-val tracking rule),
    but as a reusable function. Used both to (re)produce the baseline
    (x_override=None) and for every ROAR retrain (x_override = the zeroed-out
    feature dict for one method/threshold). Returns the state_dict of the
    best-val-accuracy epoch plus its metrics."""
    fold_seed = args.seed if seed is None else seed
    set_full_seed(fold_seed)

    fold_data = build_fold_data(fold, x_override=x_override)
    in_channels_list = [view_data.shape[1] for view_data in fold_data[0]]

    model = MultiViewGCN(
        in_channels_list=in_channels_list,
        hid_c=args.hidden_dimension,
        out_c=args.output_dimension,
        K=args.cheby_order_K,
        dropout_rate=args.dropout_rate,
        graph_mode=GRAPH_MODE,
        base_edge_weights=base_edge_weights[str(fold)] if GRAPH_MODE in LEARNABLE_GRAPH_MODES else None,
        n_subjects=number_samples,
        conv_type=CONV_TYPE,
        fusion_type=FUSION_TYPE,
    )
    if args.cuda:
        model = nn.parallel.DataParallel(model, device_ids=gpu_ids)
        model = model.cuda()

    real_model_init = model.module if isinstance(model, nn.parallel.DataParallel) else model
    if GRAPH_MODE in LEARNABLE_GRAPH_MODES:
        edge_params = list(real_model_init.edge_weight_params.parameters())
        edge_param_ids = {id(p) for p in edge_params}
        other_params = [p for p in model.parameters() if id(p) not in edge_param_ids]
        optimizer = optim.Adam([
            {"params": other_params, "lr": args.lr, "weight_decay": args.weight_decay},
            {"params": edge_params, "lr": args.lr * args.edge_lr_scale, "weight_decay": 0.0},
        ])
    else:
        optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    n_epochs = args.nEpochs if nEpochs is None else nEpochs
    best_val_acc, best_test_acc, best_epoch, best_state = 0.0, None, None, None

    for epoch in range(1, n_epochs + 1):
        (model, train_target, train_out, train_loss, val_target, val_out, val_prob,
         val_loss, test_target, test_out, test_prob, test_loss) = train(
            args, model, fold_data, dist_train[str(fold)], dist_validation[str(fold)],
            dist_test[str(fold)], optimizer)

        val_accuracy = metrics.accuracy_score(val_target, val_out)
        test_accuracy = metrics.accuracy_score(test_target, test_out)

        if val_accuracy >= best_val_acc:
            best_val_acc, best_test_acc, best_epoch = val_accuracy, test_accuracy, epoch
            real_model = model.module if isinstance(model, nn.parallel.DataParallel) else model
            best_state = copy.deepcopy(real_model.state_dict())

        if verbose and (epoch % 20 == 0 or epoch == n_epochs):
            print(f"    epoch {epoch}/{n_epochs} | val_acc={val_accuracy:.4f} | test_acc={test_accuracy:.4f}")

    return {
        "best_val_acc": best_val_acc,
        "best_test_acc": best_test_acc,
        "best_epoch": best_epoch,
        "state_dict": best_state,
        "in_channels_list": in_channels_list,
    }


def save_best_model_and_recipe(fold, result, extra_note=""):
    """(1) Your first request: saves the best model's weights for each fold.
    (2) On top of the weights, also saves a full JSON "recipe" so that anyone
    (including this notebook's own ROAR stage) can rebuild and load exactly
    this model later without guessing a single hyperparameter."""
    weight_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_best.pth")
    torch.save(result["state_dict"], weight_path)

    recipe = {
        "fold": fold,
        "seed": args.seed,
        "best_val_acc": float(result["best_val_acc"]),
        "best_test_acc": float(result["best_test_acc"]),
        "best_epoch": int(result["best_epoch"]),
        "in_channels_list": result["in_channels_list"],
        "hidden_dimension": args.hidden_dimension,
        "output_dimension": args.output_dimension,
        "cheby_order_K": args.cheby_order_K,
        "dropout_rate": args.dropout_rate,
        "graph_mode": GRAPH_MODE,
        "conv_type": CONV_TYPE,
        "fusion_type": FUSION_TYPE,
        "lr": args.lr,
        "weight_decay": args.weight_decay,
        "nEpochs": args.nEpochs,
        "edge_lr_scale": args.edge_lr_scale,
        "view_names": view_names,
        "k_neighbors_per_view": K_NEIGHBORS_PER_VIEW,
        "graph_metric": GRAPH_METRIC,
        "dist_train": [int(i) for i in dist_train[str(fold)]],
        "dist_validation": [int(i) for i in dist_validation[str(fold)]],
        "dist_test": [int(i) for i in dist_test[str(fold)]],
        "note": extra_note,
    }
    recipe_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_recipe.json")
    with open(recipe_path, "w") as f:
        json.dump(recipe, f, indent=2)

    print(f"  weights -> {weight_path}")
    print(f"  recipe  -> {recipe_path}")
    return weight_path, recipe_path


def load_best_model_for_fold(fold):
    """Loads a fold's saved recipe + weights and rebuilds the model exactly."""
    recipe_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_recipe.json")
    weight_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_best.pth")
    with open(recipe_path) as f:
        recipe = json.load(f)
    model = MultiViewGCN(
        in_channels_list=recipe["in_channels_list"],
        hid_c=recipe["hidden_dimension"],
        out_c=recipe["output_dimension"],
        K=recipe["cheby_order_K"],
        dropout_rate=recipe["dropout_rate"],
        graph_mode=recipe["graph_mode"],
        base_edge_weights=base_edge_weights[str(fold)] if recipe["graph_mode"] in LEARNABLE_GRAPH_MODES else None,
        n_subjects=number_samples,
        conv_type=recipe["conv_type"],
        fusion_type=recipe["fusion_type"],
    )
    state = torch.load(weight_path, map_location="cuda" if args.cuda else "cpu")
    model.load_state_dict(state)
    if args.cuda:
        model = model.cuda()
    return model


### 2) Load the already-saved weights per fold (no retraining)

Instead of `train_one_fold_from_scratch` (which the main notebook called here), this
cell only reads from disk: `CKPT_DIR/GCN_fold{fold}_best.pth` +
`CKPT_DIR/GCN_fold{fold}_recipe.json`.

If those files are missing for a fold, this raises a clear error (instead of silently
falling back to training from scratch) -- since the whole point of this notebook is
that no training happens here.


In [20]:
# TARGET_FOLD is the single, central place that controls which fold everything
# below (the ROAR run in cell "Main run", and the region-importance export)
# runs on. Pinned to fold 1 for the same reason as the original notebook: the
# main training loop only reseeds once before fold 1, so only fold 1's saved
# checkpoint corresponds to a run whose init you could independently verify.
# This does not affect loading -- every fold's checkpoint is loaded below --
# it only controls which fold the ROAR/explainability cells operate on.
TARGET_FOLD = 1

# Set this to True only if you want a missing fold to be trained from scratch
# instead of raising an error. Default False, since the whole point of this
# notebook is to reuse already-saved weights, not retrain.
ALLOW_RETRAIN_IF_MISSING = True

baseline_results = {}
missing_folds = []

for fold in range(1, k_fold + 1):
    recipe_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_recipe.json")
    weight_path = os.path.join(CKPT_DIR, f"GCN_fold{fold}_best.pth")

    if os.path.exists(recipe_path) and os.path.exists(weight_path):
        with open(recipe_path) as f:
            recipe = json.load(f)
        res = {
            "best_val_acc": recipe["best_val_acc"],
            "best_test_acc": recipe["best_test_acc"],
            "best_epoch": recipe["best_epoch"],
            "state_dict": torch.load(weight_path, map_location="cuda" if args.cuda else "cpu"),
            "in_channels_list": recipe["in_channels_list"],
        }
        baseline_results[fold] = res
        print(f"fold {fold}: loaded checkpoint -> {weight_path}")
        print(f"  best_val_acc={res['best_val_acc']:.4f}  best_test_acc={res['best_test_acc']:.4f}")
    elif ALLOW_RETRAIN_IF_MISSING:
        print(f"fold {fold}: no checkpoint found -- ALLOW_RETRAIN_IF_MISSING=True, training from scratch")
        res = train_one_fold_from_scratch(fold, x_override=None, seed=args.seed)
        baseline_results[fold] = res
        save_best_model_and_recipe(fold, res, extra_note="baseline (no features removed)")
    else:
        missing_folds.append(fold)

if missing_folds:
    raise FileNotFoundError(
        f"No saved checkpoint/recipe found for fold(s) {missing_folds} in {CKPT_DIR}. "
        "This notebook only loads existing weights -- it does not retrain. "
        "Run the full training notebook once (with RECOMPUTE_BASELINE=True) to "
        "generate the checkpoints, or set ALLOW_RETRAIN_IF_MISSING=True above."
    )


fold 1: loaded checkpoint -> /content/drive/MyDrive/sMRI/without_ComBat/checkpoints/GCN_fold1_best.pth
  best_val_acc=0.5972  best_test_acc=0.5951
fold 2: loaded checkpoint -> /content/drive/MyDrive/sMRI/without_ComBat/checkpoints/GCN_fold2_best.pth
  best_val_acc=0.6146  best_test_acc=0.5850
fold 3: loaded checkpoint -> /content/drive/MyDrive/sMRI/without_ComBat/checkpoints/GCN_fold3_best.pth
  best_val_acc=0.6000  best_test_acc=0.5578
fold 4: loaded checkpoint -> /content/drive/MyDrive/sMRI/without_ComBat/checkpoints/GCN_fold4_best.pth
  best_val_acc=0.5678  best_test_acc=0.6134
fold 5: loaded checkpoint -> /content/drive/MyDrive/sMRI/without_ComBat/checkpoints/GCN_fold5_best.pth
  best_val_acc=0.6289  best_test_acc=0.5735


### 3) Optional sanity check (off by default, to save time)

The main notebook retrained fold 1 from scratch here to confirm that `threshold=0` in
ROAR exactly matches the baseline. Since that is itself a full training run (up to 100
epochs), it is **off by default** here. To run this check once (just for
reassurance), set `RUN_SANITY_CHECK` to `True`.


In [21]:
RUN_SANITY_CHECK = False   # set True to retrain fold 1 once and compare to the loaded checkpoint

if RUN_SANITY_CHECK:
    check_fold = TARGET_FOLD
    res_repeat = train_one_fold_from_scratch(check_fold, x_override=None, seed=args.seed, verbose=False)

    print("baseline test acc :", baseline_results[check_fold]["best_test_acc"])
    print("repeat   test acc :", res_repeat["best_test_acc"])

    diff = abs(baseline_results[check_fold]["best_test_acc"] - res_repeat["best_test_acc"])
    if diff < 1e-9:
        print("OK: bit-exact match.")
    elif diff < 1e-3:
        print(f"OK (within GPU floating-point tolerance, diff={diff:.6f}).")
    else:
        raise AssertionError(
            f"Reproducibility check FAILED (diff={diff:.4f}). Fix this before running ROAR."
        )
else:
    print("Sanity check skipped (RUN_SANITY_CHECK=False). Using the loaded checkpoint as-is.")


Sanity check skipped (RUN_SANITY_CHECK=False). Using the loaded checkpoint as-is.


### 4) Computing attributions with 7 interpretability methods -- standard Captum API

Each "feature" here corresponds to one `(view, node, sub-feature)` column, exactly as
before. Attributions are computed on each fold's **train** subjects and averaged (to
avoid leaking val/test information into the feature ranking used later for ROAR).

**What changed from the previous version:** all 7 methods are now called through
Captum's own `captum.attr.*` classes on a single flat input tensor produced by
`MultiViewGCNCaptumWrapper` (defined below), instead of through method-specific custom
re-implementations. Concretely, this removes:
- the by-hand `gradient_shap` loop (it worked around a Captum internal bug where a
  shared random interpolation coefficient was sized to only the first view's row
  count -- see the wrapper's docstring below for the full root cause),
- the by-hand `deep_lift_shap` averaging loop (now `captum.attr.DeepLiftShap` with a
  background-sample `baselines=` tensor, exactly as documented),
- the per-subject "lump every other subject into one inert feature group" trick
  LIME/KernelSHAP needed (no longer necessary once each Captum call's input row count
  matches the model output's row count 1:1 -- see below).

**What did NOT change:**
- `MultiViewGCN` itself -- 0 lines touched.
- Training / checkpoints / ROAR retraining -- still call `MultiViewGCN` directly,
  never through the wrapper.
- The inherent approximation of IG / GradientShap / DeepLiftShap / LIME / KernelSHAP.
  Flattening the interface standardizes the *implementation*; it does not make an
  approximate method exact. `DeepLift`/`GuidedBackprop` may still fall back to default
  backprop on `ChebConv`/`GATConv` layers Captum doesn't recognize -- Captum will print
  a warning for that; it's expected and harmless.
- The wrapper assumes `GRAPH_MODE == 'static'` (asserted at import time below) -- see
  its docstring for why `'learnable'`/`'learnable_scratch'` need a different
  graph-tiling rule before the wrapper can be trusted with them.

**Before trusting any result from this section**, run the validation cell right after
it once -- it checks that `MultiViewGCNCaptumWrapper`'s output exactly matches a
direct, un-wrapped call to the loaded checkpoint on the same data. I was not able to
execute this notebook myself (no access to your data/GPU/checkpoints), so this check
is not optional -- it is the only proof that the refactor below is a pure reshape and
not a silent change in what is actually being explained.


### Fix: per-subject target class (was fixed to class 1 for everyone)

**Before:** `forward_func` always returned `torch.exp(out)[:, 1]`, i.e. every subject's
attribution (ASD or control) was computed with respect to "the probability of class 1".
For class-0 subjects, that meant "which features push the model incorrectly toward
ASD", not necessarily the best discriminative biomarkers.

**Now:** `forward_func` returns the full probability vector (both classes), and
`target=` on each Captum call specifies that each subject is explained with respect to
**its own class**, not one fixed class for everyone. `target_mode` has three settings:

- `"true"` (default) -- each subject's true label.
- `"predicted"` -- the class the model itself predicted for that subject.
- an integer (e.g. `1`) -- the old behavior, kept only to compare against/reproduce
  earlier results.

The rest of the pipeline (ranking, ROAR, `zero_top_k`, etc.) is unchanged -- only this
one point (target-class selection in attribution) was fixed.


In [22]:
from captum.attr import (
    IntegratedGradients, DeepLift, DeepLiftShap, GradientShap, GuidedBackprop, Lime, KernelShap,
)

XAI_METHODS = [
    "integrated_gradients", "guided_backprop", "deep_lift",
    "deep_lift_shap", "gradient_shap", "lime", "kernel_shap",
]

# =============================================================================
# Standard-Captum-API wrapper for MultiViewGCN
# =============================================================================
# WHY THIS EXISTS
# ----------------
# MultiViewGCN.forward(data_list, edge_index_list, edgenet_input_list,
# batch_vec_list) takes 4 separate arguments (3 of them per-view lists), and its
# node-feature tensors are stacked over (subject x node) rows rather than one row
# per subject. Captum's official attribution classes are built around a single
# differentiable "inputs" tensor whose leading dimension is the batch of
# independent examples being explained, matching forward_func's output rows 1:1.
# The previous version of this cell worked around that mismatch with several
# method-specific custom re-implementations (a hand-rolled GradientShap, a
# hand-rolled DeepLiftShap averaging loop, and a per-subject "lump everyone else
# into one inert feature group" trick for LIME/KernelSHAP).
#
# The wrapper below removes the mismatch at its source instead of working around
# it downstream: it exposes MultiViewGCN as a plain forward(x_flat) ->
# probabilities module, where x_flat has shape (n_rows, total_flat_features) and
# EVERY ROW IS ONE SUBJECT. Internally it reshapes/splits x_flat back into the
# model's native per-view, per-node layout, re-tiles the block-diagonal graph to
# match however many subject-rows Captum handed it (a real subject count, or a
# multiple of it from IG's n_steps / GradientShap's n_samples interpolation
# replicas -- the wrapper does not need to know which; it only reads
# x_flat.shape[0]), and calls MultiViewGCN.forward() completely unchanged.
#
# WHAT THIS DOES NOT CHANGE
#   - MultiViewGCN itself: 0 lines touched.
#   - Training / checkpoints / ROAR retraining: 0 lines touched -- they all call
#     MultiViewGCN directly, never through this wrapper.
#   - The approximation inherent to IG / GradientShap / DeepLiftShap / LIME /
#     KernelSHAP: exactly as approximate as Captum's own implementation of each
#     method. This wrapper only standardizes the *interface*, not the *math* of
#     an approximate method.
#
# KNOWN LIMITATION -- read before using with GRAPH_MODE != 'static'
#   MultiViewGCN.forward(), when graph_mode is 'learnable' or 'learnable_scratch',
#   computes
#       edge_weight = self.edge_weight_params[i].repeat(self.n_subjects)
#   where self.n_subjects was fixed at model-construction time to the WHOLE
#   fold's subject count. That line hard-codes the batch size the model will
#   accept; it does not adapt to however many subject-rows are actually passed in
#   a given forward call. The wrapper below re-tiles the graph to match the
#   ACTUAL number of rows in each Captum call, which only agrees with the
#   model's hard-coded edge_weight length when graph_mode == 'static' (there,
#   edge_weight comes from edgenet_input_list, which the wrapper is free to
#   re-tile to any size). This notebook currently runs with GRAPH_MODE = 'static'
#   (set above); the assertion below turns a silent wrong-shape bug into a clear
#   error if that ever changes without revisiting this wrapper's graph-tiling
#   logic for the learnable modes.
assert GRAPH_MODE == 'static', (
    "MultiViewGCNCaptumWrapper assumes GRAPH_MODE == 'static' (edge_weight "
    "supplied per call, not hard-coded to the fold's subject count via "
    "self.n_subjects). Re-derive the wrapper's graph-tiling logic before using "
    "it with graph_mode='learnable' or 'learnable_scratch'."
)


class MultiViewGCNCaptumWrapper(nn.Module):
    """Standard single-tensor-in / single-tensor-out view of MultiViewGCN.

    forward(x_flat) -> class probabilities (n_rows, n_classes), where x_flat has
    shape (n_rows, total_flat_features) and every row is one subject (or one
    interpolation/perturbation replica of a subject -- Captum decides how many
    rows a given call contains; the wrapper only needs x_flat.shape[0]).
    """

    def __init__(self, real_model, view_names, n_nodes_per_view, n_subfeat_per_view,
                 base_edge_index_np, base_edge_weight_np, device):
        super().__init__()
        # Registered as a submodule (not just captured in a closure) so Captum's
        # hook-based methods (GuidedBackprop, DeepLift, DeepLiftShap) can
        # discover real_model's actual nn.ReLU layers by traversing
        # self.modules() -- a plain closure could not offer that.
        self.real_model = real_model
        self.view_names = view_names
        self.n_nodes_per_view = n_nodes_per_view
        self.n_subfeat_per_view = n_subfeat_per_view
        self.device = device
        # base_edge_index_np / base_edge_weight_np: {view: SINGLE-COPY edge_index /
        # edge_weight}, straight out of build_covariance_graph -- the same graph
        # object this fold's tiled data[str(fold)] graph was built from. Never
        # touched by ROAR/perturbation; only node features change.
        self.base_edge_index_np = base_edge_index_np
        self.base_edge_weight_np = base_edge_weight_np
        self._graph_cache = {}   # n_rows -> (edge_index_list, edge_weight_list, batch_vec_list) on device

        # Fix, once, the exact column bijection between a flat row and the
        # model's native per-view/per-node layout. Every flatten/split below
        # reuses this mapping so the two directions can never drift out of sync.
        self._flat_slices = {}
        offset = 0
        for v in view_names:
            width = n_nodes_per_view[v] * n_subfeat_per_view[v]
            self._flat_slices[v] = (offset, offset + width)
            offset += width
        self.total_flat_features = offset

    def flatten_subject_features(self, per_view_subject_major):
        """per_view_subject_major: {view: tensor (n_subjects, n_nodes, n_subfeat)},
        i.e. view_node_features's own layout. Returns the flat
        (n_subjects, total_flat_features) tensor this wrapper's forward() expects."""
        n_subjects = next(iter(per_view_subject_major.values())).shape[0]
        chunks = [per_view_subject_major[v].reshape(n_subjects, -1) for v in self.view_names]
        return torch.cat(chunks, dim=1)

    def _split_flat_to_views(self, x_flat):
        """Inverse of flatten_subject_features, generalized to any row count.
        Returns {view: (n_rows * n_nodes_v, n_subfeat_v)} in the SAME
        subject-major row order MultiViewGCN.forward() / X_BATCHED already use
        (row block [r*n_nodes_v : (r+1)*n_nodes_v] = row r's own nodes)."""
        n_rows = x_flat.shape[0]
        out = {}
        for v in self.view_names:
            lo, hi = self._flat_slices[v]
            n_nodes = self.n_nodes_per_view[v]
            n_subfeat = self.n_subfeat_per_view[v]
            out[v] = x_flat[:, lo:hi].reshape(n_rows * n_nodes, n_subfeat)
        return out, n_rows

    def _graph_for_n_rows(self, n_rows):
        if n_rows not in self._graph_cache:
            edge_index_list, edge_weight_list, batch_vec_list = [], [], []
            for v in self.view_names:
                n_nodes = self.n_nodes_per_view[v]
                ei_b, ew_b = tile_graph_for_batch(
                    self.base_edge_index_np[v], self.base_edge_weight_np[v], n_nodes, n_rows)
                bv_np = np.repeat(np.arange(n_rows), n_nodes).astype(np.int64)
                edge_index_list.append(torch.from_numpy(ei_b).to(self.device))
                edge_weight_list.append(torch.from_numpy(ew_b).float().to(self.device))
                batch_vec_list.append(torch.from_numpy(bv_np).to(self.device))
            self._graph_cache[n_rows] = (edge_index_list, edge_weight_list, batch_vec_list)
        return self._graph_cache[n_rows]

    def forward(self, x_flat):
        per_view, n_rows = self._split_flat_to_views(x_flat)
        edge_index_list, edge_weight_list, batch_vec_list = self._graph_for_n_rows(n_rows)
        data_list = [per_view[v] for v in self.view_names]
        out = self.real_model(data_list, edge_index_list, edge_weight_list, batch_vec_list)
        return torch.exp(out)   # real_model ends in nn.LogSoftmax; exponentiate to
                                 # hand Captum plain class probabilities, exactly
                                 # like the previous forward_func did.


BASELINE_STRATEGIES = ('zero', 'mean', 'random_noise', 'real_sample_shuffle')
# 'zero'               : all-zero baseline -- the standard default for standardized
#                        features (zero already coincides with the dataset mean
#                        under this pipeline's StandardScaler).
# 'mean'               : the actual per-column average computed directly from this
#                        fold's real data -- the classic "expected value" baseline
#                        from the original IG / SHAP papers.
# 'random_noise'       : small Gaussian noise, N(0, 0.05) -- a good choice as a
#                        *background distribution* (DeepLiftShap / GradientShap
#                        redraw it per sample for genuine diversity); a less
#                        standard choice as the single fixed baseline of plain
#                        DeepLift/IG.
# 'real_sample_shuffle': a REAL subject's flat feature vector, drawn from this
#                        fold -- every baseline value genuinely occurs in the data,
#                        which is the background-sample strategy SHAP itself
#                        recommends when a real background dataset is available.

IG_BASELINE_STRATEGY            = 'zero'           # Integrated Gradients (single baseline per subject)
DEEPLIFT_BASELINE_STRATEGY      = 'zero'           # DeepLift              (single baseline per subject)
DEEPLIFT_SHAP_BASELINE_STRATEGY = 'random_noise'   # DeepLiftShap (background sample, passed via native baselines=)
GRADIENT_SHAP_BASELINE_STRATEGY = 'zero'           # GradientShap (background sample; diversity also comes from
                                                    # n_samples + stdevs interpolation, same as Captum's own default)


def make_flat_baseline(x_flat_all, kind, n_draws=1):
    """x_flat_all: (n_subjects, total_flat_features) -- this fold's REAL flat
    data, used as the background pool for 'mean'/'real_sample_shuffle'.
    n_draws == 1  -> (n_subjects, total_flat_features), one baseline row per
                      subject (IntegratedGradients / DeepLift: baselines must
                      align 1:1 with inputs).
    n_draws  > 1  -> (n_draws, total_flat_features), a background SAMPLE
                      (DeepLiftShap / GradientShap's native `baselines=`
                      broadcasts this set of draws against every input row
                      internally -- this is their documented usage, not a
                      custom loop)."""
    device, dtype = x_flat_all.device, x_flat_all.dtype
    n_features = x_flat_all.shape[1]
    if kind == 'zero':
        shape = x_flat_all.shape if n_draws == 1 else (n_draws, n_features)
        return torch.zeros(shape, device=device, dtype=dtype)
    elif kind == 'mean':
        mean_row = x_flat_all.detach().mean(dim=0, keepdim=True)
        reps = x_flat_all.shape[0] if n_draws == 1 else n_draws
        return mean_row.expand(reps, -1).clone()
    elif kind == 'random_noise':
        shape = x_flat_all.shape if n_draws == 1 else (n_draws, n_features)
        return 0.05 * torch.randn(shape, device=device, dtype=dtype)
    elif kind == 'real_sample_shuffle':
        n_subjects = x_flat_all.shape[0]
        n_pick = n_subjects if n_draws == 1 else n_draws
        idx = torch.randint(0, n_subjects, (n_pick,), device=device)
        return x_flat_all.detach()[idx].clone()
    else:
        raise ValueError(f"baseline kind must be one of {BASELINE_STRATEGIES}, got: {kind!r}")


def _flat_attr_to_views(mean_attr_flat, wrapper, n_nodes_per_view, n_subfeat_per_view):
    """mean_attr_flat: 1D np.ndarray, length wrapper.total_flat_features, already
    |.|-and-averaged over subjects. Splits it back into {view: (n_nodes, n_subfeat)}
    -- the same shape compute_attributions has always returned -- reusing the
    wrapper's own column bookkeeping so this can never drift out of sync with
    flatten_subject_features / _split_flat_to_views."""
    result = {}
    for v in wrapper.view_names:
        lo, hi = wrapper._flat_slices[v]
        result[v] = mean_attr_flat[lo:hi].reshape(n_nodes_per_view[v], n_subfeat_per_view[v])
    return result


def _build_wrapper_for_fold(fold):
    """Loads this fold's checkpoint and wraps it. Recovers the single-copy
    (un-tiled) graph directly from data[str(fold)]'s already-tiled arrays --
    block 0 of a tile_graph_for_batch(..., n_subjects) output IS the single-copy
    graph, so there is no need to rebuild it or store it separately."""
    model = load_best_model_for_fold(fold)
    model.eval()
    device = next(model.parameters()).device

    n_nodes_per_view = {v: view_node_features[v].shape[1] for v in view_names}
    n_subfeat_per_view = {v: view_node_features[v].shape[2] for v in view_names}
    _, _, edge_index_list_np, edgenet_input_list_np = data[str(fold)]

    base_edge_index_np, base_edge_weight_np = {}, {}
    for i, v in enumerate(view_names):
        n_edges_per_copy = edge_index_list_np[i].shape[1] // number_samples
        base_edge_index_np[v] = edge_index_list_np[i][:, :n_edges_per_copy]
        base_edge_weight_np[v] = edgenet_input_list_np[i][:n_edges_per_copy].reshape(-1)

    wrapper = MultiViewGCNCaptumWrapper(
        model, view_names, n_nodes_per_view, n_subfeat_per_view,
        base_edge_index_np, base_edge_weight_np, device)
    wrapper.eval()
    if args.cuda:
        wrapper = wrapper.cuda()
    return wrapper, n_nodes_per_view, n_subfeat_per_view, device


def compute_attributions(fold, method, target_mode="true", subjects="train",
                          n_ig_steps=32, n_shap_samples=20,
                          lime_n_samples=500, kernelshap_n_samples=500,
                          internal_batch_size=None):
    """Standard-Captum-API version: every method below is called through the
    OFFICIAL captum.attr class on the single flat input tensor produced by
    MultiViewGCNCaptumWrapper -- no per-method custom re-implementation. Returns
    {view: np.ndarray shape (n_nodes, n_subfeat)} = mean |attribution| over the
    chosen subjects for this fold's saved best model, exactly like the previous
    version, so every downstream cell (ranking, ROAR, CSV export) is unchanged.

    target_mode:
      - "true"      (default): each subject's own ground-truth label.
      - "predicted": each subject's own model-predicted class.
      - an int (e.g. 1): the same fixed class for every subject, kept only for
        comparing against older/paper-style results.
    """
    wrapper, n_nodes_per_view, n_subfeat_per_view, device = _build_wrapper_for_fold(fold)

    _, labels_onehot, _, _ = data[str(fold)]
    true_class = np.argmax(labels_onehot, axis=1)

    per_view_subject_major = {v: torch.from_numpy(view_node_features[v]).float() for v in view_names}
    if args.cuda:
        per_view_subject_major = {v: t.to(device) for v, t in per_view_subject_major.items()}
    x_flat_all = wrapper.flatten_subject_features(per_view_subject_major)
    x_flat_all = x_flat_all.clone().requires_grad_(True)

    if target_mode == "true":
        target_idx = true_class
    elif target_mode == "predicted":
        with torch.no_grad():
            base_probs = wrapper(x_flat_all)
        target_idx = torch.argmax(base_probs, dim=1).detach().cpu().numpy()
    elif isinstance(target_mode, int):
        target_idx = np.full(number_samples, target_mode, dtype=np.int64)
    else:
        raise ValueError(f"unknown target_mode {target_mode!r}")
    target_t = torch.from_numpy(target_idx).long().to(device)

    if subjects == "train":
        idx = np.array(sorted(dist_train[str(fold)]))
    elif subjects == "all":
        idx = np.arange(number_samples)
    else:
        idx = np.array(subjects)

    x_flat = x_flat_all[idx]
    target_sub = target_t[idx]

    if internal_batch_size is None:
        internal_batch_size = max(len(idx), 1) * 4

    if method == "integrated_gradients":
        baselines = make_flat_baseline(x_flat_all, IG_BASELINE_STRATEGY)[idx]
        attrs_flat = IntegratedGradients(wrapper).attribute(
            x_flat, baselines=baselines, target=target_sub,
            n_steps=n_ig_steps, internal_batch_size=internal_batch_size)

    elif method == "guided_backprop":
        attrs_flat = GuidedBackprop(wrapper).attribute(x_flat, target=target_sub)

    elif method == "deep_lift":
        baselines = make_flat_baseline(x_flat_all, DEEPLIFT_BASELINE_STRATEGY)[idx]
        attrs_flat = DeepLift(wrapper).attribute(x_flat, baselines=baselines, target=target_sub)

    elif method == "deep_lift_shap":
        # Native captum.attr.DeepLiftShap: baselines is a BACKGROUND SAMPLE
        # (n_shap_samples rows), broadcast against every input row internally --
        # this replaces the previous by-hand "loop DeepLift n_shap_samples times
        # and average" with Captum's own documented DeepLiftShap usage.
        background = make_flat_baseline(x_flat_all, DEEPLIFT_SHAP_BASELINE_STRATEGY, n_draws=n_shap_samples)
        attrs_flat = DeepLiftShap(wrapper).attribute(x_flat, baselines=background, target=target_sub)

    elif method == "gradient_shap":
        # Native captum.attr.GradientShap. The previous version had to
        # reimplement this method by hand because Captum's own GradientShap
        # draws ONE random interpolation coefficient sized to inputs[0]'s row
        # count and reuses it for every tensor in a multi-tensor `inputs` tuple
        # -- broken whenever the tuple's tensors have different row counts, as
        # our 3 per-view tensors did. With a SINGLE flat tensor there is only
        # one row count, so that internal assumption is never violated and the
        # official class can be used directly.
        background = make_flat_baseline(x_flat_all, GRADIENT_SHAP_BASELINE_STRATEGY, n_draws=n_shap_samples)
        attrs_flat = GradientShap(wrapper).attribute(
            x_flat, baselines=background, target=target_sub, n_samples=n_shap_samples, stdevs=0.05)

    elif method in ("lime", "kernel_shap"):
        # Looped per subject, exactly like every other method here explains
        # each subject w.r.t. its OWN class -- this loop is inherent to
        # Captum's Lime/KernelShap API (forward_func must reduce to one scalar
        # per .attribute() call, so a batch of independent subjects can't be
        # explained in a single call), not a multi-view workaround. What DID go
        # away versus the previous version: no more "lump every other subject
        # into one inert feature group" trick, and no more
        # return_input_shape=False workaround -- both were only needed because
        # the old forward_func always processed the WHOLE fold's block-diagonal
        # graph per call. Here, x_subj is exactly one row, so wrapper(x_subj)
        # only ever builds/uses that ONE subject's own small single-copy graph
        # (via _graph_for_n_rows(1)), which is both simpler and cheaper.
        n_samples = lime_n_samples if method == "lime" else kernelshap_n_samples
        Explainer = Lime if method == "lime" else KernelShap
        explainer = Explainer(wrapper)
        summed = np.zeros(wrapper.total_flat_features, dtype=np.float64)
        feature_mask = torch.arange(wrapper.total_flat_features, device=device).view(1, -1)
        for subj in idx:
            x_subj = x_flat_all[subj:subj + 1]
            baseline_subj = torch.zeros_like(x_subj)
            coefs = explainer.attribute(
                x_subj, baselines=baseline_subj, feature_mask=feature_mask,
                target=int(target_t[subj].item()), n_samples=n_samples)
            summed += np.abs(coefs.detach().cpu().numpy().reshape(-1))
        mean_attr = summed / len(idx)
        return _flat_attr_to_views(mean_attr, wrapper, n_nodes_per_view, n_subfeat_per_view)

    else:
        raise ValueError(f"unknown method {method!r}")

    attr_np = np.abs(attrs_flat.detach().cpu().numpy())
    mean_attr = attr_np.mean(axis=0)
    return _flat_attr_to_views(mean_attr, wrapper, n_nodes_per_view, n_subfeat_per_view)


### 4a) Ranking + CSV-export helpers (restored -- were missing in this notebook)
These two functions were present in the earlier version of this notebook but were dropped when it was refactored to the standard-Captum-API wrapper above, which caused `NameError: name 'get_feature_ranking' is not defined` in section 6b. They are unchanged from the earlier version and work on `compute_attributions()`'s output unmodified (`{view: np.ndarray (n_nodes, n_subfeat)}`), so no other cell needs to change.

In [23]:
def get_feature_ranking(attr_dict):
    """Flattens {view: (n_nodes, n_subfeat)} into a list of
    (view, node_idx, subfeat_idx, score) sorted by |score| descending --
    this is the ranking ROAR removes features by, top-first. Works on the
    attr_dict returned by compute_attributions() above (unchanged format
    between the old and the standard-Captum-API version of that function)."""
    ranking = []
    for view, arr in attr_dict.items():
        n_nodes, n_subfeat = arr.shape
        for node_idx in range(n_nodes):
            for subfeat_idx in range(n_subfeat):
                ranking.append((view, node_idx, subfeat_idx, float(arr[node_idx, subfeat_idx])))
    ranking.sort(key=lambda t: -abs(t[3]))
    return ranking


def save_importance_table(fold, method, attr_dict):
    """Saves the important features -- at the (node, sub-feature) level."""
    rows = []
    for view, arr in attr_dict.items():
        n_nodes, n_subfeat = arr.shape
        node_names = view_node_names[view]
        suffixes = VIEW_CONFIGS[view]["suffixes"]
        for node_idx in range(n_nodes):
            for subfeat_idx in range(n_subfeat):
                rows.append({
                    "fold": fold, "method": method, "view": view,
                    "roi_name": node_names[node_idx],
                    "sub_feature": suffixes[subfeat_idx],
                    "importance": arr[node_idx, subfeat_idx],
                })
    df = pd.DataFrame(rows).sort_values("importance", ascending=False)
    out_path = os.path.join(XAI_DIR, f"feature_importance_fold{fold}_{method}.csv")
    df.to_csv(out_path, index=False)
    print(f"  saved -> {out_path}")
    return df


### 4b) Mandatory sanity check -- run this once before trusting any attribution above

This does not use Captum at all. It only proves that `MultiViewGCNCaptumWrapper` is a
pure reshape of the same data `MultiViewGCN` already sees -- i.e. that the
flatten/split logic introduces no semantic drift -- by comparing the wrapper's output
against a direct, un-wrapped call to the SAME loaded checkpoint on the SAME fold,
using the model's own native tiled graph (`data[str(fold)]`, `BATCH_VEC`) as ground
truth.

**I was not able to run this notebook myself** (no access to your data, GPU, or saved
checkpoints), so this check is not optional -- it is the only real proof that the
refactor above did not silently change what is being explained. If it fails, stop and
fix the mismatch before using any attribution/ranking/ROAR result computed through the
wrapper.


In [24]:
def validate_wrapper_matches_model(fold, atol=1e-5):
    wrapper, n_nodes_per_view, n_subfeat_per_view, device = _build_wrapper_for_fold(fold)
    model = wrapper.real_model

    data_list_np, _, ei_np, ew_np = data[str(fold)]
    data_list = [torch.from_numpy(d).float().to(device) for d in data_list_np]
    edge_index_list = [torch.from_numpy(ei).to(device) for ei in ei_np]
    edgenet_input_list = [torch.from_numpy(ew).float().to(device) for ew in ew_np]
    batch_vec_list = [torch.from_numpy(BATCH_VEC[v]).to(device) for v in view_names]
    with torch.no_grad():
        direct_out = torch.exp(model(data_list, edge_index_list, edgenet_input_list, batch_vec_list))

    per_view_subject_major = {v: torch.from_numpy(view_node_features[v]).float().to(device) for v in view_names}
    x_flat = wrapper.flatten_subject_features(per_view_subject_major)
    with torch.no_grad():
        wrapper_out = wrapper(x_flat)

    max_diff = (direct_out - wrapper_out).abs().max().item()
    print(f"fold {fold}: max |direct_model_output - wrapper_output| = {max_diff:.3e}")
    if max_diff > atol:
        raise AssertionError(
            f"Wrapper output diverges from the direct model call by {max_diff:.3e} "
            f"(> atol={atol}). Do NOT trust attributions from this wrapper until this "
            "is fixed -- it means the flatten/split reshape logic is not a pure "
            "re-layout of the same data."
        )
    print("  OK: wrapper output matches the direct, un-wrapped model call.")
    return max_diff


_ = validate_wrapper_matches_model(TARGET_FOLD)


fold 1: max |direct_model_output - wrapper_output| = 1.192e-07
  OK: wrapper output matches the direct, un-wrapped model call.


### 5) ROAR core: zero out the top-ranked columns + retrain from scratch at each threshold

`zero_top_k` always starts from the **original/untouched** `X_BATCHED` (never
from the previous threshold's output), exactly matching the paper: *"for each
threshold, we retrain the model from scratch on this modified dataset"* — so
thresholds are independent of each other and rebuilt independently. Graph
topology is never touched, only node features.

In [25]:
ROAR_THRESHOLDS = [0.0, 0.01, 0.05, 0.075, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]  # same as the paper


def zero_top_k(view_arrays, ranking, frac):
    """view_arrays: dict view -> ORIGINAL np.ndarray (n_subjects*n_nodes, n_subfeat).
    Zeroes the top `frac` fraction of ranked (view, node, subfeat) columns for
    EVERY subject -- the paper's 'replace with an uninformative value (zero)'
    rule -- and returns a fresh deep copy (never mutates view_arrays)."""
    out = {v: view_arrays[v].copy() for v in view_arrays}
    if frac <= 0:
        return out
    n_remove = int(round(frac * len(ranking)))
    n_nodes_per_view = {v: view_node_features[v].shape[1] for v in view_names}
    for view, node_idx, subfeat_idx, _score in ranking[:n_remove]:
        n_nodes = n_nodes_per_view[view]
        rows = np.arange(node_idx, out[view].shape[0], n_nodes)   # this node's row, every subject
        out[view][rows, subfeat_idx] = 0.0
    return out


def run_roar(method_name, ranking, fold, thresholds=ROAR_THRESHOLDS, nEpochs=None):
    original_arrays = {v: X_BATCHED[v] for v in view_names}   # untouched originals
    rows = []
    for frac in thresholds:
        x_override = zero_top_k(original_arrays, ranking, frac)
        res = train_one_fold_from_scratch(fold, x_override=x_override, seed=args.seed,
                                           nEpochs=nEpochs, verbose=False)
        rows.append({"method": method_name, "fold": fold, "threshold": frac,
                      "test_acc": res["best_test_acc"], "val_acc": res["best_val_acc"]})
        print(f"  [{method_name}] fold={fold} threshold={frac:.2f} -> test_acc={res['best_test_acc']:.4f}")
    return rows


### 6) Comparison setup (shared by both the random baseline and the real methods)


In [26]:
# --- Pick which explainability methods to run below (this list IS the selection --
# just remove/add entries, no other cell needs to change): ---
ALL_METHODS_TO_RUN = ["lime", "kernel_shap",]
# , "guided_backprop", "deep_lift", "gradient_shap"]
# Enable the heavier methods (higher compute cost) by adding these:
# ALL_METHODS_TO_RUN += ["deep_lift_shap", "lime", "kernel_shap"]

ROAR_FOLDS = [TARGET_FOLD]   # set in cell above; for the paper's full scale: list(range(1, k_fold + 1))
ROAR_EPOCHS = 100          # matches args.nEpochs (= the main loop's / baseline's epoch count above),
                           # so every threshold -- including 0.00 -- is trained under the exact same
                           # protocol as the main loop. This costs more compute than the original
                           # ROAR_EPOCHS=40 shortcut (14 thresholds x 100 epochs instead of x 40 --
                           # roughly 2.5x the training time for this cell), which is the tradeoff for
                           # a protocol-consistent comparison instead of a faster approximate one.

roar_rows = []      # shared by both sections below -- rows from "random" and from every real method
attr_cache = {}      # (fold, method) -> ranking, filled by the real-methods section further down
failed_methods = []   # (fold, method, error) -- so one broken method doesn't kill the whole run


# --- Explainability sample-count / step-count knobs (previously buried as
# hard-coded defaults inside compute_attributions() in the cell above, and
# NEVER actually passed in by the main run loop below -- so they were always
# silently stuck at 32 / 20 / 200 / 200 no matter what. Now they're visible
# and configurable here, and the run loop below passes them explicitly. ---
N_IG_STEPS            = 32     # IntegratedGradients: n_steps
N_SHAP_SAMPLES         = 20     # GradientShap / DeepLiftShap: background n_draws / n_samples
LIME_N_SAMPLES         = 200    # Lime: n_samples (perturbed samples per subject)
KERNELSHAP_N_SAMPLES   = 200    # KernelShap: n_samples (perturbed samples per subject)
# NOTE: your flattened per-subject input has ~2137 columns (45x7 + 148x9 + 70x7,
# per the "Load sMRI" cell's printed node/sub-feature counts), and LIME/KernelSHAP
# treat every column as its own feature group (feature_mask = arange(total_flat_features)).
# 200 perturbed samples to fit a linear surrogate over that many groups is very
# low and likely to give unstable / noisy per-feature attributions -- consider
# raising LIME_N_SAMPLES / KERNELSHAP_N_SAMPLES (e.g. into the low thousands) if
# runtime allows, and treat results from the current defaults with caution.


### 6a) Random baseline -- fully independent, runs BEFORE any attribution method

This section is deliberately **fully separate and independent** from any attribution
method: it builds the random ranking directly from each view's own shape
(`view_node_features[view].shape`), not by shuffling some method's output. That way,
even if none of the methods in the next section run or all of them fail, this section
still works on its own.

So the removed percentages at each threshold are **comparable** to the real methods,
this random list's length exactly matches the total number of (view, node,
sub-feature) columns the real methods also work on (both are built from the same
source -- `view_node_features`'s shape); only the removal ORDER is fully random
(`np.random.default_rng(args.seed).shuffle`), not ranked by importance.


In [27]:
INCLUDE_RANDOM_BASELINE = False  # set False to skip this comparison entirely


def get_random_feature_ranking(seed):
    """Builds the full (view, node_idx, subfeat_idx, score) list directly from
    view_node_features' shapes -- completely independent of any attribution
    method or attr_cache -- then shuffles it into a random order. `score` is
    unused (kept only for format-compatibility with get_feature_ranking); what
    matters is the random ORDER, since zero_top_k removes ranking[:n_remove]."""
    ranking = []
    for view in view_names:
        n_nodes = view_node_features[view].shape[1]
        n_subfeat = view_node_features[view].shape[2]
        for node_idx in range(n_nodes):
            for subfeat_idx in range(n_subfeat):
                ranking.append((view, node_idx, subfeat_idx, 0.0))
    rng = np.random.default_rng(seed)
    rng.shuffle(ranking)
    return ranking


if INCLUDE_RANDOM_BASELINE:
    for fold in ROAR_FOLDS:
        print(f"\n=== random baseline (fully independent), fold {fold} ===")
        random_ranking = get_random_feature_ranking(seed=args.seed)
        roar_rows += run_roar("random", random_ranking, fold, nEpochs=ROAR_EPOCHS)


### 6b) Attribution methods with ROAR (runs after the random baseline above)


In [ ]:
for fold in ROAR_FOLDS:
    for method in ALL_METHODS_TO_RUN:
        print(f"\n=== computing {method} attributions, fold {fold} ===")
        import gc; gc.collect(); torch.cuda.empty_cache()  # OOM fix: clear GPU memory before each method
        try:
            attr_dict = compute_attributions(
                fold, method,
                n_ig_steps=N_IG_STEPS,
                n_shap_samples=N_SHAP_SAMPLES,
                lime_n_samples=LIME_N_SAMPLES,
                kernelshap_n_samples=KERNELSHAP_N_SAMPLES,
            )
            ranking = get_feature_ranking(attr_dict)
            attr_cache[(fold, method)] = ranking
            save_importance_table(fold, method, attr_dict)
            roar_rows += run_roar(method, ranking, fold, nEpochs=ROAR_EPOCHS)
        except Exception as e:
            print(f"  !! {method} failed on fold {fold}, skipping it and continuing: {e!r}")
            failed_methods.append((fold, method, repr(e)))
            continue

if failed_methods:
    print(f"\n(skipped due to errors: {failed_methods})")



=== computing lime attributions, fold 1 ===
  saved -> /content/drive/MyDrive/sMRI/without_ComBat/explainability/feature_importance_fold1_lime.csv
  [lime] fold=1 threshold=0.00 -> test_acc=0.6098
  [lime] fold=1 threshold=0.01 -> test_acc=0.5610
  [lime] fold=1 threshold=0.05 -> test_acc=0.5561
  [lime] fold=1 threshold=0.07 -> test_acc=0.5707
  [lime] fold=1 threshold=0.10 -> test_acc=0.5317
  [lime] fold=1 threshold=0.20 -> test_acc=0.5512
  [lime] fold=1 threshold=0.30 -> test_acc=0.5366
  [lime] fold=1 threshold=0.40 -> test_acc=0.5317
  [lime] fold=1 threshold=0.50 -> test_acc=0.5024


In [ ]:
roar_df = pd.DataFrame(roar_rows)
roar_csv_path = os.path.join(XAI_DIR, "roar_results.csv")
roar_df.to_csv(roar_csv_path, index=False)
print(f"\nsaved ROAR results (random baseline + real methods) -> {roar_csv_path}")
roar_df.head()


### 7) Comparison plot (equivalent to the paper's Fig. 5) and picking the "most reliable method"

In [ ]:
# plt.figure(figsize=(9, 6))
# for method, g in roar_df.groupby("method"):
#     g = g.sort_values("threshold")
#     plt.plot(g["threshold"] * 100, g["test_acc"] * 100, marker="o", label=method)
# plt.xlabel("Percent of features removed")
# plt.ylabel("Test accuracy (%)")
# plt.title("ROAR: accuracy vs. % features removed, by explainability method")
# plt.legend()
# plt.grid(color="black", linestyle="-", linewidth=0.5)
# plot_path = os.path.join(XAI_DIR, "roar_comparison.png")
# plt.savefig(plot_path, dpi=150, bbox_inches="tight")
# plt.show()
# print(f"saved plot -> {plot_path}")


# def area_under_curve(df, method):
#     g = df[df.method == method].sort_values("threshold")
#     return np.trapz(g["test_acc"], g["threshold"])   # smaller area = steeper/earlier drop


# scores = {m: area_under_curve(roar_df, m) for m in roar_df.method.unique()}
# best_method = min(scores, key=scores.get)
# print("\nArea under the accuracy-vs-removed curve (lower = steeper drop = more reliable):")
# for m, s in sorted(scores.items(), key=lambda kv: kv[1]):
#     print(f"  {m:22s} {s:.4f}")
# print(f"\n=> Most reliable method (steepest drop, matching the paper\'s ROAR criterion): {best_method}")


### 8) Final brain-region ranking (ROI level) using the best method — equivalent to the paper's Table 3

In [ ]:
# def export_region_importance(method, fold, top_n=25):
#     attr_dict = compute_attributions(fold, method)
#     rows = []
#     for view, arr in attr_dict.items():
#         node_names = view_node_names[view]
#         node_importance = np.abs(arr).sum(axis=1)   # sum |importance| over sub-features -> one score per ROI
#         for node_idx, roi_name in enumerate(node_names):
#             rows.append({"view": view, "roi_name": roi_name, "importance": float(node_importance[node_idx])})
#     region_df = pd.DataFrame(rows).sort_values("importance", ascending=False)
#     out_path = os.path.join(XAI_DIR, f"important_regions_{method}_fold{fold}.csv")
#     region_df.to_csv(out_path, index=False)
#     print(f"saved -> {out_path}\n")
#     print(region_df.head(top_n).to_string(index=False))
#     return region_df


# ROI_FOLD = ROAR_FOLDS[0]
# import gc; gc.collect(); torch.cuda.empty_cache()  # OOM fix: clear GPU memory before final region-importance pass
# if best_method != "random":
#     region_df = export_region_importance(best_method, ROI_FOLD)
# else:
#     print("The best method came out as 'random', meaning none of the real methods beat random removal -- "
#           "before finalizing the region ranking, revisit the epoch count / thresholds, or the n_samples "
#           "of the perturbation-based methods.")


**Important note on Brodmann Area mapping:** your ROIs come from the
FreeSurfer atlas (aseg / Desikan-Killiany aparc / wmparc), not the AAL atlas
the paper used for fMRI — these two atlases don't share region names, and I'm
not going to guess a FreeSurfer→Brodmann mapping from memory (too high a risk
of getting it wrong). If you want the ROIs in the CSV above mapped to Brodmann
Areas, use a verified Desikan-Killiany↔Brodmann correspondence table from the
literature and apply it to `region_df`. I'm happy to write that step too if
you provide/confirm the reference correspondence table you want to use.